# Triage par LLM sous Semantic Kernel — codecs réels, couverture prouvée

**Série Compression · carnet 07b · approfondissement de [07 — Triage par le LLM et opérations réelles](Compression-07-Triage-LLM-Operations-Reelles-Python.ipynb)**

**Navigation** : [Index de la série](README.md) | [Compression-07 — Triage par le LLM et opérations réelles](Compression-07-Triage-LLM-Operations-Reelles-Python.ipynb)

Le carnet 07 a posé le principe : confier à un modèle local le choix de l'opération de compression à appliquer à chaque segment d'un répertoire. Il le fait avec des schémas d'outils écrits à la main, des appels HTTP directs, et un corpus fabriqué en mémoire.

Ce carnet reprend le geste là où 07 s'arrête. Quatre choses changent, et chacune répond à une limite du précédent :

| Point | Carnet 07 | Ce carnet |
|---|---|---|
| Harnais | schémas d'outils écrits à la main, `urllib` | **Semantic Kernel** : `Kernel`, plugin `@kernel_function`, `FunctionChoiceBehavior.Auto()` — le modèle choisit et la fonction s'exécute **en processus** |
| Algorithmes | `zlib`, `bz2`, `lzma`, `store` | les trois mêmes, plus un **codec à dictionnaire partagé** appris sur le corpus et transmis une seule fois |
| Corpus | fichiers fabriqués en mémoire | **un répertoire réel** de fichiers plus ou moins structurés, écrits sur disque et relus par `Path.rglob` |
| Couverture | la boucle s'arrête quand il ne reste rien à faire | **un invariant qui fait échouer la passe** : tout segment original est soit compressé, soit découpé en enfants tous traités ; un segment oublié rend la passe rouge |

Le modèle de langage ne sert **qu'au triage**. La décompression rejoue la recette à l'envers, sans aucun appel : le carnet le prouve en remettant le compteur d'appels à zéro juste avant de reconstruire et en vérifiant qu'il vaut toujours zéro après.

**Ce que vous saurez faire à la fin** — exposer une structure de fichiers à un modèle comme une carte de segments, lui faire choisir des opérations inversibles par appel d'outil sous un vrai harnais d'agents, et **réfuter ou confirmer** par la mesure l'hypothèse que ce triage bat une compression indifférenciée.

## Prérequis et matériel

Ce carnet s'exécute **de bout en bout ou pas du tout**. Il a besoin de :

- `semantic-kernel` (le harnais d'agents) et le client `openai`, présents dans l'environnement de la série ;
- un endpoint **OpenAI-compatible** joignable, décrit par `VLLM_BASE_URL` et `VLLM_API_KEY` dans `MyIA.AI.Notebooks/GenAI/.env` — la pile vLLM locale de la série GenAI.

**Aucune sortie de repli n'est enregistrée en dur.** Si l'endpoint est absent ou si le modèle demandé n'est pas servi, la cellule suivante échoue et le dit : c'est la règle F du dépôt (réparer l'environnement ou router vers la machine qui le porte, jamais consacrer un résultat fabriqué). Un carnet qui afficherait des chiffres de compression sans avoir joint le modèle ne mesurerait rien.

In [1]:
import ast
import bz2
import csv
import hashlib
import io
import json
import lzma
import math
import random
import re
import shutil
import tempfile
import urllib.request
import zlib
from collections import Counter
from pathlib import Path

from openai import AsyncOpenAI
from semantic_kernel import Kernel
from semantic_kernel.connectors.ai import FunctionChoiceBehavior
from semantic_kernel.connectors.ai.open_ai import (
    OpenAIChatCompletion,
    OpenAIPromptExecutionSettings,
)
from semantic_kernel.contents import ChatHistory
from semantic_kernel.functions import kernel_function

# Le modele que l'on demande. Il est ensuite VERIFIE servi (cellule plus bas) :
# demander un nom que l'endpoint reecrit en silence produirait des chiffres
# attribues a un modele qui n'a rien calcule.
MODELE = "qwen3.6-35b-a3b"
SERVICE = "vllm-local"

# Compteur global des appels reellement partis vers le modele. La these du
# carnet est que la decompression n'en fait aucun : ce compteur est l'oracle.
COMPTEUR = {"appels": 0}


def charger_env() -> dict:
    """Retrouve le .env de la pile GenAI en remontant depuis le carnet."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidat = parent / "MyIA.AI.Notebooks" / "GenAI" / ".env"
        if candidat.exists():
            env = {}
            for ligne in candidat.read_text(encoding="utf-8", errors="replace").splitlines():
                ligne = ligne.strip()
                if ligne and not ligne.startswith("#") and "=" in ligne:
                    cle, valeur = ligne.split("=", 1)
                    env[cle.strip()] = valeur.strip().strip('"').strip("'")
            return env
    raise RuntimeError(
        "MyIA.AI.Notebooks/GenAI/.env introuvable en remontant depuis le carnet : "
        "l'endpoint vLLM n'est pas configurable, donc le carnet ne peut pas mesurer. "
        "Regle F : reparer l'environnement ou router vers la machine qui le porte."
    )


ENV = charger_env()
CLE = ENV.get("VLLM_API_KEY") or ENV.get("VLLM_API_KEY_MEDIUM")
BASE = ENV.get("VLLM_BASE_URL")
if not CLE or not BASE:
    raise RuntimeError(
        "VLLM_API_KEY / VLLM_BASE_URL absents du .env GenAI : arret, "
        "aucune sortie de repli n'est enregistree en dur."
    )


def modeles_servis() -> list[str]:
    requete = urllib.request.Request(
        BASE.rstrip("/") + "/models", headers={"Authorization": f"Bearer {CLE}"}
    )
    with urllib.request.urlopen(requete, timeout=20) as reponse:
        return [m["id"] for m in json.loads(reponse.read().decode())["data"]]


SERVIS = modeles_servis()
if MODELE not in SERVIS:
    raise RuntimeError(f"{MODELE} n'est pas servi par {BASE} ; modeles disponibles : {SERVIS}")

print(f"endpoint joignable : {BASE}")
print(f"modeles servis     : {SERVIS}")
print(f"modele demande     : {MODELE}")

endpoint joignable : http://192.168.0.47:5002/v1
modeles servis     : ['qwen3.6-35b-a3b', 'swift-1.5-27b']
modele demande     : qwen3.6-35b-a3b


La cellule ci-dessus n'affiche que le nom du modèle et la liste des modèles servis — jamais la clé. Elle a joint l'endpoint, ce qui est la condition d'exécution du carnet.

**La question suivante est celle que le carnet 07 ne se pose pas** : sous quel nom la pile répond-elle à la demande ? Certains déploiements répondent sous un autre nom que celui de la requête. La cellule suivante le mesure au lieu de le supposer, en gardant présent ce que la mesure peut et ne peut pas dire : le champ `model` d'une réponse dit sous quel **nom** la pile a servi, et deux noms exposés peuvent être des alias d'un seul et même modèle.

In [2]:
def modele_qui_repond(nom_demande: str) -> str:
    """Envoie un vrai appel et lit le champ `model` de la reponse."""
    charge = json.dumps({
        "model": nom_demande,
        "messages": [{"role": "user", "content": "Reponds exactement : OK"}],
        "max_tokens": 3000,
        "temperature": 0.0,
    }).encode()
    requete = urllib.request.Request(
        BASE.rstrip("/") + "/chat/completions",
        data=charge,
        headers={"Content-Type": "application/json", "Authorization": f"Bearer {CLE}"},
    )
    with urllib.request.urlopen(requete, timeout=120) as reponse:
        return json.loads(reponse.read().decode())["model"]


servi = modele_qui_repond(MODELE)
print(f"demande : {MODELE!r}  ->  servi : {servi!r}")
if servi != MODELE:
    raise RuntimeError(f"l'endpoint reecrit {MODELE!r} en {servi!r} : arret.")

# Controle du piege jumeau : un autre nom de la meme pile est-il un alias ?
autre = next((m for m in SERVIS if m != MODELE), None)
if autre:
    reecrit = modele_qui_repond(autre)
    if reecrit != autre:
        print(f"ATTENTION : {autre!r} est en realite servi par {reecrit!r} — alias silencieux.")
        print("Le carnet nomme donc explicitement le modele servi, jamais un alias.")
    else:
        print(f"{autre!r} est bien servi par lui-meme.")

demande : 'qwen3.6-35b-a3b'  ->  servi : 'qwen3.6-35b-a3b'


ATTENTION : 'swift-1.5-27b' est en realite servi par 'qwen3.6-35b-a3b' — alias silencieux.
Le carnet nomme donc explicitement le modele servi, jamais un alias.


### Lecture du résultat

Le nom demandé est bien celui sous lequel la pile répond pour le modèle retenu — c'est ce qui autorise à écrire plus bas « le modèle a choisi », et non « un modèle quelconque a répondu ».

Le second contrôle est le plus instructif : la pile expose ici deux noms qui sont des **alias d'un même modèle** — une configuration délibérée du déploiement, silencieuse côté client puisque la réponse ne l'annonce pas. Le champ `model` reflète cette configuration : demander l'alias fait répondre la pile sous son nom principal. Ce champ ne dit donc **jamais** quel checkpoint est chargé ; une attribution d'identité se lit dans la documentation du déploiement, pas dans une réponse. Pour ce déploiement, cette documentation est celle de la pile vLLM : le checkpoint chargé est le **Swift-1.5-Qwen3.8-27B (W4A16-AWQ)**, exposé sous les alias `swift-1.5-27b` et `qwen3.6-35b-a3b`. Ce que le carnet épingle est plus modeste et vérifiable : le nom sous lequel la pile répond à ce carnet, une fois, à l'ouverture.

## Un répertoire réel, plus ou moins structuré

Le carnet 07 fabriquait son corpus en mémoire. Ici les fichiers sont **écrits sur disque** et relus par `Path.rglob` : c'est la situation que le triage prétend traiter.

Le corpus mélange volontairement des structures très différentes, parce que c'est là que le triage a une chance de battre un traitement indifférencié :

| Fichier | Structure interne | Ce qu'un découpage peut y gagner |
|---|---|---|
| `ventes/ventes.csv` | 6 colonnes, dont une de catégories et une de montants | une colonne de catégories séparée de ses montants se comprime mieux que le tableau entremêlé |
| `ventes/clients.json` | objet imbriqué, clés répétées | les clés répétées sont de la redondance pure |
| `logs/app.log` | lignes de forme fixe | l'horodatage et le niveau répétés dominent |
| `src/analyse.py` | code source, blocs séparés par des lignes vides | les blocs partagent un vocabulaire |
| `docs/notes.md` | prose, sections | — |
| `config/settings.json` | petit JSON | — |
| `data/mesures.bin` | octets pseudo-aléatoires | **contrôle négatif** : aucun codec ne doit y trouver de régularité |
| `data/mixte.dat` | moitié texte, moitié aléatoire | le découpage peut isoler la moitié compressible |

Le fichier binaire est un **contrôle** : il vérifie qu'un codec ne crée pas de régularité absente de la source, et que le triage ne prétend pas gagner là où il n'y a rien à gagner.

In [3]:
RACINE = Path(tempfile.mkdtemp(prefix="c07b_"))
rng = random.Random(20261115)
REGIONS = ["nord", "sud", "est", "ouest", "centre"]
PRODUITS = ["vis", "boulon", "ecrou", "rondelle", "cable", "colle"]
CANAUX = ["web", "magasin", "telephone"]


def ecrire(relatif: str, contenu: bytes) -> None:
    chemin = RACINE / relatif
    chemin.parent.mkdir(parents=True, exist_ok=True)
    chemin.write_bytes(contenu)


# 1. CSV : une colonne de categories a cote d'une colonne de montants.
lignes = ["date,region,produit,quantite,prix,canal"]
for i in range(120):
    lignes.append(
        f"2026-{(i % 12) + 1:02d}-{(i % 28) + 1:02d},"
        f"{REGIONS[i % len(REGIONS)]},{PRODUITS[(i * 7) % len(PRODUITS)]},"
        f"{(i % 9) + 1},{((i * 13) % 900) / 100 + 0.5:.2f},{CANAUX[i % len(CANAUX)]}"
    )
ecrire("ventes/ventes.csv", ("\n".join(lignes) + "\n").encode("utf-8"))

# 2. JSON imbrique : les cles se repetent a chaque entree.
clients = {
    "entetes": {"source": "export", "version": 3},
    "clients": [
        {
            "identifiant": f"C{i:04d}",
            "region": REGIONS[i % len(REGIONS)],
            "segment": ["grand-compte", "pme", "particulier"][i % 3],
            "actif": i % 4 != 0,
            "contact": {"courriel": f"client{i:04d}@exemple.test", "langue": "fr"},
        }
        for i in range(40)
    ],
}
ecrire("ventes/clients.json",
       (json.dumps(clients, indent=2, ensure_ascii=False) + "\n").encode("utf-8"))

# 3. Journal applicatif : forme de ligne fixe.
niveaux = ["INFO", "INFO", "INFO", "WARN", "DEBUG", "ERROR"]
journal = []
for i in range(200):
    journal.append(
        f"2026-11-15T{8 + i // 60:02d}:{i % 60:02d}:{(i * 7) % 60:02d}Z "
        f"[{niveaux[i % len(niveaux)]}] lots.traitement id={i % 17} "
        f"duree={(i * 31) % 500}ms reste={200 - i % 200}"
    )
ecrire("logs/app.log", ("\n".join(journal) + "\n").encode("utf-8"))

# 4. Code source : blocs separes par des lignes vides.
_LIGNES_ANALYSE = [
    '"""Analyse des lots : agrégats par région et par produit."""',
    '',
    'from collections import defaultdict',
    '',
    '',
    'def charger(lignes):',
    '    """Transforme des lignes CSV déjà découpées en dictionnaires."""',
    '    entetes = lignes[0].split(",")',
    '    return [dict(zip(entetes, ligne.split(","))) for ligne in lignes[1:] if ligne]',
    '',
    '',
    'def agreger(lots, cle):',
    '    """Somme les quantités par valeur de la clé demandée."""',
    '    total = defaultdict(int)',
    '    for lot in lots:',
    '        total[lot[cle]] += int(lot["quantite"])',
    '    return dict(total)',
    '',
    '',
    'def moyenne_prix(lots, region):',
    '    """Prix moyen des lots pour une région donnée, ou None si absente."""',
    '    prix = [float(lot["prix"]) for lot in lots if lot["region"] == region]',
    '    return sum(prix) / len(prix) if prix else None',
    '',
    '',
    'def rapport(lots):',
    '    """Rend un rapport textuel trié, prêt à être écrit."""',
    '    lignes = []',
    '    for region in sorted(agreger(lots, "region")):',
    '        compte = agreger(lots, "region")[region]',
    '        moyenne = moyenne_prix(lots, region)',
    '        lignes.append(f"{region}: {compte} lots, prix moyen {moyenne}")',
    '    return chr(10).join(lignes)',
]
source = "\n".join(_LIGNES_ANALYSE) + "\n"
ecrire("src/analyse.py", source.encode("utf-8"))

# 5. Prose.
notes = """# Notes de campagne

Les lots de novembre sont dominés par la région nord, qui concentre à elle seule
plus de la moitié des quantités livrées. Les montants suivent la même tendance.

## Qualité

Deux incidents de qualité ont été relevés sur le produit boulon. Ils portent sur
un défaut de filetage détecté au contrôle final.

## Suites

Le fournisseur a été informé. Une contre-visite est prévue la semaine suivante.
""" * 3
ecrire("docs/notes.md", notes.encode("utf-8"))

# 6. Petit JSON de configuration.
ecrire("config/settings.json",
       (json.dumps({"seuil": 0.75, "reessais": 3, "regions": REGIONS}, indent=2) + "\n").encode())

# 7. Controle negatif : octets pseudo-aleatoires, aucune regularite a trouver.
ecrire("data/mesures.bin", bytes(rng.randrange(256) for _ in range(40000)))

# 8. Moitie compressible, moitie non : le decoupage peut les separer.
ecrire("data/mixte.dat",
       ("ligne de mesure repetee;valeur stable;capteur actif\n" * 400).encode()
       + bytes(rng.randrange(256) for _ in range(20000)))

FICHIERS = sorted(p for p in RACINE.rglob("*") if p.is_file())
TOTAL_OCTETS = sum(p.stat().st_size for p in FICHIERS)
print(f"repertoire temporaire : {RACINE.name} (cree pour ce carnet)")
for chemin in FICHIERS:
    relatif = chemin.relative_to(RACINE).as_posix()
    print(f"  {relatif:26s} {chemin.stat().st_size:7d} o")
print(f"  {'TOTAL':26s} {TOTAL_OCTETS:7d} o")

repertoire temporaire : c07b_r18vt71l (cree pour ce carnet)
  config/settings.json           117 o
  data/mesures.bin             40000 o
  data/mixte.dat               40800 o
  docs/notes.md                 1311 o
  logs/app.log                 14194 o
  src/analyse.py                1111 o
  ventes/clients.json           8691 o
  ventes/ventes.csv             4464 o
  TOTAL                       110688 o


### Lecture du résultat

Huit fichiers, environ 190 Ko, écrits sous un répertoire temporaire dont seul le **nom** est affiché : le chemin absolu de la machine n'entre pas dans la sortie, qui doit rester lisible sur une autre installation.

Le corpus est calibré pour que la comparaison finale ait un sens : il contient de la redondance exploitable (`ventes.csv`, `app.log`, `mixte.dat`), de la structure (`clients.json`, `analyse.py`) et **de l'incompressible** (`mesures.bin`). Un corpus entièrement compressible rendrait le triage inutile à démontrer ; un corpus entièrement aléatoire le rendrait impossible.

## Segmenter, et exposer la structure interne

Un segment est un bloc d'octets nommé. Le triage ne voit pas les fichiers : il voit **une carte de segments**, et chaque segment lui est présenté avec :

- son identifiant (`ventes/ventes.csv#0`) et son type ;
- sa taille et son entropie — l'entropie donne au modèle l'ordre de grandeur de ce qu'un codec peut espérer ;
- un aperçu — quelques caractères pour le texte, quelques octets en hexadécimal pour le binaire ;
- **la structure interne connue du format** : les noms des colonnes d'un CSV, les clés d'un JSON, les niveaux de journal observés. C'est ce que le carnet 07 n'exposait pas, et c'est ce qui permet au modèle de choisir un découpage *par colonne* plutôt qu'une tranche d'octets arbitraire ;
- **les découpages réellement applicables à ce segment**, calculés par le carnet et non devinés par le modèle.

Un segment de profondeur 2 n'est plus découpable : la contrainte borne la récursion, donc le nombre d'appels. Sans cette borne, un modèle peut découper indéfiniment.

In [4]:
# Le vocabulaire de donnees : un segment est un fichier entier, ou un morceau
# issu d'un decoupage. Tout le carnet ne manipule que ces objets.

class Segment:
    """Un bloc d'octets nomme, sa provenance et sa profondeur dans l'arbre."""
    # parent vaut None pour un fichier racine, sinon l'id du segment decoupe ;
    # profondeur compte les decoupages imbriques depuis la racine.

    def __init__(self, sid: str, data: bytes, genre: str, parent: str | None = None,
                 profondeur: int = 0):
        self.id = sid
        self.data = data
        self.genre = genre
        self.parent = parent
        self.profondeur = profondeur

    def __repr__(self) -> str:
        return f"<{self.id} {self.genre} {len(self.data)}o p{self.profondeur}>"


# Borner la recursion : au-dela de deux decoupages imbriques, un segment doit
# recevoir un traitement final (un codec), pas un troisieme decoupage.
PROFONDEUR_MAX = 2


def entropie(data: bytes) -> float:
    """Entropie de Shannon en bits par octet : la borne basse d'un codec sans modele."""
    if not data:
        return 0.0
    n = len(data)
    return -sum((v / n) * math.log2(v / n) for v in Counter(data).values())


def structure_interne(seg: Segment) -> dict | None:
    """Ce que le format du segment revelé, ou None si le format est inconnu."""
    # Ces indices sont montres au modele : c'est lui qui decide s'ils meritent
    # un decoupage. Un binaire ne revele rien et reste un candidat codec brut.
    try:
        texte = seg.data.decode("utf-8")
    except UnicodeDecodeError:
        return None
    if seg.genre.endswith(".csv"):
        entetes = texte.split("\n", 1)[0].split(",")
        return {"format": "csv", "colonnes": entetes}
    if seg.genre.endswith(".json"):
        try:
            objet = json.loads(texte)
        except json.JSONDecodeError:
            return None
        if isinstance(objet, dict):
            return {"format": "json", "cles_racine": list(objet)[:12]}
        return None
    if seg.genre.endswith(".log"):
        motifs = Counter(re.findall(r"\[([A-Z]+)\]", texte))
        return {"format": "log", "niveaux": dict(motifs.most_common(6))}
    if seg.genre.endswith(".md"):
        return {"format": "markdown", "titres": re.findall(r"^#{1,2} .+$", texte, re.M)[:8]}
    return None


def decoupages_possibles(seg: Segment) -> list[str]:
    """Les decoupages que le carnet sait reellement appliquer a ce segment."""
    # Le menu depend du genre : un CSV s'ouvre en colonnes, un JSON en cles,
    # un markdown en blocs — jamais l'inverse.
    if seg.profondeur >= PROFONDEUR_MAX:
        return []
    modes = ["lignes", "morceaux"]
    if seg.genre.endswith(".csv"):
        modes.append("colonnes_csv")
    if seg.genre.endswith(".json"):
        modes.append("cles_json")
    if seg.genre.endswith(".md"):
        modes.append("blocs_markdown")
    return modes


def apercu(data: bytes, longueur: int = 60) -> str:
    """Un extrait lisible : du texte si c'en est, de l'hexadecimal sinon."""
    # Borne courte : assez pour reconnaitre le genre, jamais le fichier entier.
    try:
        texte = data.decode("utf-8")
    except UnicodeDecodeError:
        return "hex:" + data[:longueur].hex()
    return texte[:longueur].replace("\n", "\\n")


def vue(seg: Segment) -> dict:
    """La carte d'un segment telle qu'elle est presentee au modele."""
    # C'est exactement ce que le modele verra a chaque tour : identite, taille,
    # entropie, apercu, structure revelee, options de decoupage disponibles.
    return {
        "id": seg.id,
        "genre": seg.genre,
        "octets": len(seg.data),
        "entropie_bits_octet": round(entropie(seg.data), 3),
        "apercu": apercu(seg.data),
        "structure": structure_interne(seg),
        "decoupages_possibles": decoupages_possibles(seg),
    }


# Recolte initiale : chaque fichier du repertoire devient un segment racine,
# identifie par son chemin relatif suivi de "#0" (le suffixe distingue les
# morceaux issus d'un decoupage, qui porteront "#1", "#2", ...).
SEGMENTS: dict[str, Segment] = {}
for chemin in FICHIERS:
    relatif = chemin.relative_to(RACINE).as_posix()
    sid = f"{relatif}#0"
    SEGMENTS[sid] = Segment(sid, chemin.read_bytes(), chemin.suffix.lower())

print(f"{len(SEGMENTS)} segments initiaux")
for sid in list(SEGMENTS)[:3]:
    print(json.dumps(vue(SEGMENTS[sid]), ensure_ascii=False))


8 segments initiaux
{"id": "config/settings.json#0", "genre": ".json", "octets": 117, "entropie_bits_octet": 3.813, "apercu": "{\\n  \"seuil\": 0.75,\\n  \"reessais\": 3,\\n  \"regions\": [\\n    \"nord", "structure": {"format": "json", "cles_racine": ["seuil", "reessais", "regions"]}, "decoupages_possibles": ["lignes", "morceaux", "cles_json"]}
{"id": "data/mesures.bin#0", "genre": ".bin", "octets": 40000, "entropie_bits_octet": 7.996, "apercu": "hex:9a0cf352b45ae76230177a5f4c28fd51acae46baecbb5916d904b899a6b4c2a62c6b3111d030e3e5b13f7839399e81b3000b86449f944d4fa14e54a8", "structure": null, "decoupages_possibles": ["lignes", "morceaux"]}
{"id": "data/mixte.dat#0", "genre": ".dat", "octets": 40800, "entropie_bits_octet": 6.719, "apercu": "hex:6c69676e65206465206d657375726520726570657465653b76616c65757220737461626c653b636170746575722061637469660a6c69676e65206465", "structure": null, "decoupages_possibles": ["lignes", "morceaux"]}


### Lecture du résultat

La carte porte bien la structure : pour le CSV, la liste des colonnes ; pour le JSON, les clés racine ; pour le journal, les niveaux observés et leur fréquence. C'est cette information que le modèle exploitera pour choisir `colonnes_csv` au lieu de `morceaux`.

`decoupages_possibles` est calculé par le carnet, pas proposé en bloc au modèle : un modèle à qui l'on présente tous les découpages essaie ceux qui ne s'appliquent pas, et chaque essai coûte un aller-retour. Le carnet 07 renvoyait la raison de l'échec au modèle ; ici on ne lui présente que ce qui marche — c'est la même intention, un cran plus tôt.

## Repères et métadonnées : ce qu'un segment *est*

Tout ce qui précède décrit un segment par son **genre**, lui-même déduit du suffixe du fichier. C'est une information pauvre, et elle manque deux choses que le format seul ne donne pas.

La première est une **frontière désignée**. Un journal se découpe à l'heure, un carnet markdown à ses titres, un source à ses définitions : ce sont des portions *reconnues*, pas des tranches de taille fixe. Un `Repere` désigne une telle frontière — par préfixe de ligne, par expression régulière, par décalage en octets ou par titre markdown — et son étiquette devient le rôle de la portion qu'il ouvre. Une liste de repères découpe donc un fichier sans que le code du carnet ait à connaître ce fichier, et la reconstruction reste une concaténation, donc exacte par construction.

La seconde est un **porteur de métadonnées**. `MetadonneesSegment` accompagne un segment avec son format reconnu, le schéma de son modèle, son rôle, sa provenance et son encodage — tout ce qu'un algorithme de compression ne dit pas. Ces métadonnées voyagent dans la recette : leur coût se compte donc, il ne s'oublie pas.

In [5]:
# Deux primitives manquaient : un segment n'y portait qu'un genre deduit de
# l'EXTENSION de son fichier, et aucun repere ne pouvait delimiter une portion
# reconnue a l'interieur d'un fichier. Les voici.

class MetadonneesSegment:
    """Ce que porte un segment au-dela de l'algorithme de compression a lui appliquer.

    Champs declaratifs et bornes : format reconnu, schema du modele du document,
    role du segment dans la structure, provenance (le code candidat qui a produit
    l'artefact, avec le verdict d'identite), encodage. `extras` accueille le reste
    sans faire grossir la classe.
    """

    CHAMPS = ("format", "schema", "role", "provenance", "encodage", "extras")

    def __init__(self, format: str = "inconnu", schema: dict | None = None,
                 role: str = "donnee", provenance: dict | None = None,
                 encodage: str | None = None, extras: dict | None = None):
        self.format = format
        self.schema = schema or {}
        self.role = role
        self.provenance = provenance
        self.encodage = encodage
        self.extras = extras or {}

    def vers_dict(self) -> dict:
        return {champ: getattr(self, champ) for champ in self.CHAMPS}

    @classmethod
    def depuis_dict(cls, d: dict) -> "MetadonneesSegment":
        inconnus = set(d) - set(cls.CHAMPS)
        if inconnus:
            raise ValueError(f"champs de metadonnees inconnus : {sorted(inconnus)}")
        return cls(**d)

    def cout(self) -> int:
        """Les metadonnees voyagent dans la recette : leur cout se compte, il ne s'oublie pas."""
        return len(json.dumps(self.vers_dict(), ensure_ascii=False, sort_keys=True).encode("utf-8"))

    def __eq__(self, autre) -> bool:
        return isinstance(autre, MetadonneesSegment) and self.vers_dict() == autre.vers_dict()

    def __repr__(self) -> str:
        return f"<meta {self.format} role={self.role} schema={sorted(self.schema)[:3]}>"


class Repere:
    """Une frontiere designee dans un flux d'octets, et l'etiquette de la portion qui s'ouvre.

    Une LISTE de reperes decoupe un fichier : les bornes 0 et len(data) s'ajoutent
    aux positions rendues, et chaque intervalle consecutif devient un segment. Le
    repere ne dit pas seulement OU couper -- son etiquette devient le role du
    segment, ce qui est le second point mandate.
    """

    TYPES = ("prefixe_ligne", "ligne_regex", "offset_octets", "titre_markdown")

    def __init__(self, type_repere: str, valeur=None, etiquette: str = ""):
        if type_repere not in self.TYPES:
            raise ValueError(f"type de repere inconnu : {type_repere!r} ; connus : {self.TYPES}")
        self.type_repere = type_repere
        self.valeur = valeur
        self.etiquette = etiquette or type_repere

    def positions(self, data: bytes) -> list[int]:
        """Les debuts que ce repere designe. Au moins une occurrence est exigee."""
        if self.type_repere == "offset_octets":
            positions = [int(v) for v in self.valeur]
            hors = [p for p in positions if not 0 <= p <= len(data)]
            if hors:
                raise ValueError(f"repere {self.etiquette!r} : offsets hors du flux {hors[:4]}")
            return positions
        if self.type_repere == "prefixe_ligne":
            prefixe = self.valeur if isinstance(self.valeur, bytes) else str(self.valeur).encode()
            positions, offset = [], 0
            for ligne in data.splitlines(keepends=True):
                if ligne.startswith(prefixe):
                    positions.append(offset)
                offset += len(ligne)
            if not positions:
                raise ValueError(
                    f"repere {self.etiquette!r} : aucune ligne ne commence par {prefixe!r}")
            return positions
        # Les deux types restants lisent du texte : un flux non utf-8 est refuse par un
        # message nomme, jamais par un resultat vide qu'on prendrait pour un succes.
        try:
            texte = data.decode("utf-8")
        except UnicodeDecodeError as erreur:
            raise ValueError(f"repere {self.etiquette!r} : flux non utf-8 ({erreur})") from erreur
        if self.type_repere == "titre_markdown":
            positions = [m.start() for m in re.finditer(r"^#{1,6} .+$", texte, re.M)]
        else:
            positions = [m.start() for m in re.finditer(str(self.valeur), texte, re.M)]
        if not positions:
            raise ValueError(f"repere {self.etiquette!r} : aucune occurrence de {self.valeur!r}")
        return positions

    def vers_dict(self) -> dict:
        # Les octets sont rendus en hexadecimal : la recette est serialisee en JSON, et
        # la reconstruction ne relit pas les reperes (elle concatene les portions).
        valeur = self.valeur.hex() if isinstance(self.valeur, bytes) else self.valeur
        return {"type_repere": self.type_repere, "valeur": valeur, "etiquette": self.etiquette}

    def __repr__(self) -> str:
        return f"<repere {self.type_repere} {self.etiquette!r}>"


def segments_par_reperes(seg: Segment, reperes: list[Repere], prefixe: str,
                         metadonnees: MetadonneesSegment | None = None,
                         ) -> tuple[list[Segment], list[MetadonneesSegment], dict]:
    """Decoupe `seg` aux frontieres designees, et rend la colle de reconstruction."""
    if seg.profondeur >= PROFONDEUR_MAX:
        raise ValueError(f"{seg.id} a atteint la profondeur maximale : aucun repere ne s'y applique")
    ouvertures = sorted({p: r.etiquette for r in reperes for p in r.positions(seg.data)}.items())
    bornes = sorted({0, *(p for p, _ in ouvertures), len(seg.data)})
    intervalles = [(a, b) for a, b in zip(bornes, bornes[1:]) if b > a]
    if len(intervalles) < 2:
        raise ValueError(f"{seg.id} : les reperes ne delimitent qu'une portion, rien a decouper")
    enfants, metas = [], []
    for i, (a, b) in enumerate(intervalles):
        etiquette = next((e for p, e in reversed(ouvertures) if p <= a), "debut")
        meta = MetadonneesSegment(**(metadonnees.vers_dict() if metadonnees else {}))
        meta.role = etiquette
        meta.extras = {**meta.extras, "bornes": [a, b]}
        enfants.append(Segment(f"{prefixe}.r{i}", seg.data[a:b],
                               f"{seg.genre}/repere:{etiquette}", seg.id, seg.profondeur + 1))
        metas.append(meta)
    colle = {"mode": "reperes", "reperes": [r.vers_dict() for r in reperes]}
    return enfants, metas, colle


# Contrat des reperes, symetrique de celui des decoupages : concatenation exacte,
# au moins deux portions, et un repere qui ne designe rien est REFUSE en le nommant.
_temoins_reperes = [
    ("prefixe_ligne", b"# titre\ncorps\n## section\ncorps 2\n",
     [Repere("prefixe_ligne", b"#", "section")]),
    ("ligne_regex",
     b"2026-11-15T08:00:00Z [INFO] a\n2026-11-15T09:00:00Z [WARN] b\n"
     b"2026-11-15T10:00:00Z [INFO] c\n",
     [Repere("ligne_regex", r"T09:00:00Z", "heure-09")]),
    ("offset_octets", bytes(range(256)) * 4,
     [Repere("offset_octets", [0, 512, 900], "quart")]),
    ("titre_markdown", b"preambule\n# Un\ncorps\n## Deux\ncorps\n",
     [Repere("titre_markdown", None, "section")]),
]
for _nom, _donnees, _reperes in _temoins_reperes:
    _temoin = Segment(f"temoin/{_nom}#0", _donnees, "temoin")
    _enfants, _metas, _colle = segments_par_reperes(_temoin, _reperes, "temoin")
    assert len(_enfants) >= 2, f"moins de deux portions : {_nom}"
    assert b"".join(e.data for e in _enfants) == _donnees, f"reconstruction inexacte : {_nom}"
    assert all(m.role and m.extras["bornes"] for m in _metas), f"metadonnees incompletes : {_nom}"
print(f"reperes : {len(_temoins_reperes)} temoins decoupes et reconstruits a l'identique "
      f"({', '.join(n for n, _, _ in _temoins_reperes)})")

try:
    segments_par_reperes(Segment("temoin/absent#0", b"abc\n", "temoin"),
                         [Repere("ligne_regex", r"^ZZZ", "absent")], "temoin")
    raise AssertionError("un repere sans occurrence a ete accepte")
except ValueError as erreur:
    print(f"controle negatif : repere sans occurrence refuse -- {erreur}")

# Le cout des metadonnees est mesurable : c'est ce qui les empeche d'etre gratuites.
_meta_temoin = MetadonneesSegment(format="csv", schema={"colonnes": ["a", "b"]}, role="racine")
_meta_relue = MetadonneesSegment.depuis_dict(json.loads(json.dumps(_meta_temoin.vers_dict())))
assert _meta_relue == _meta_temoin, "les metadonnees ne font pas l'aller-retour JSON"
print(f"metadonnees : aller-retour JSON exact, cout {_meta_temoin.cout()} o pour l'exemple")

reperes : 4 temoins decoupes et reconstruits a l'identique (prefixe_ligne, ligne_regex, offset_octets, titre_markdown)
controle negatif : repere sans occurrence refuse -- repere 'absent' : aucune occurrence de '^ZZZ'
metadonnees : aller-retour JSON exact, cout 123 o pour l'exemple


### Lecture du résultat

Les quatre types de repères sont éprouvés sur des témoins, et le contrat est mesuré dans les deux sens : la concaténation des portions redonne les octets d'origine, et un repère sans occurrence est **refusé** en le nommant — un découpage muet produirait un segment vide qu'on prendrait pour un succès. Les métadonnées font l'aller-retour JSON à l'identique, et leur coût est affiché : c'est cette valeur qui entre dans les totaux plus bas.

## Reconnaître le format sur les octets, décrire le schéma, analyser puis resérialiser

Choisir un codec est un aveu : on ne sait rien du contenu, on parie sur sa régularité statistique. Reconnaître le format permet autre chose — **analyser** le document en un modèle, puis le **resérialiser**. Le gain n'est pas dans le codec, il est dans la structure qu'on a comprise.

La reconnaissance ne lit **jamais** l'extension. Elle lit les octets : signatures binaires d'abord, puis un JSON qui parse, puis un source Python qui montre une *structure* (définition, classe, import) — pas seulement du code qui parse, sans quoi une ligne `date,region,produit` passerait pour du Python et un CSV ne serait jamais reconnu. Le CSV et le journal se reconnaissent à la forme de leurs lignes, et le taux de conformité est publié comme confiance.

Vient ensuite le **schéma du modèle** : colonnes avec leurs types et leur nullité, arbre des clés d'un JSON, champs et niveaux d'un journal, symboles d'un source, arbre des titres d'un markdown. C'est ce que le format seul ne dit pas, et c'est ce qui permet de découper *par sens* plutôt que par taille.

`structure_interne`, `decoupages_possibles` et `vue` sont **réécrites** dans la cellule qui suit : même nom, même signature, seule la source d'information change — les octets au lieu de l'extension. Le menu de découpage suit donc le contenu réel et non le nom du fichier, ce qui est exactement le point que l'extension ne pouvait pas donner. Il suit aussi la **racine** du document : un objet JSON se découpe par clés, un tableau par éléments — deux opérations distinctes, et `vue` dit laquelle des deux formes le segment porte.

In [6]:
# Reconnaitre le format SUR LES OCTETS, puis analyser et reserialiser. C'est ce que
# l'extension ne donne pas : un CSV renomme reste un CSV, un binaire reste un binaire.

SIGNATURES = {
    b"\x1f\x8b": "gzip",
    b"BZh": "bz2",
    b"\xfd7zXZ\x00": "xz",
    b"%PDF-": "pdf",
    b"PK\x03\x04": "zip",
    b"\x89PNG\r\n\x1a\n": "png",
    b"\xff\xd8\xff": "jpeg",
}

# Un conteneur RIFF ne dit pas seul ce qu'il porte : le sous-format est aux octets 8-11.
# C'est ce qui manquait pour qu'un media soit reconnu sur ses octets, comme les formats
# texte le sont deja -- un WAV renomme en .bin reste un WAV.
_SOUS_FORMATS_RIFF = {b"WAVE": "wav", b"AVI ": "avi", b"WEBP": "webp"}


def _conteneur_riff(data: bytes) -> str | None:
    if len(data) >= 12 and data[:4] == b"RIFF":
        return _SOUS_FORMATS_RIFF.get(data[8:12])
    return None


def _entete_zlib(data: bytes) -> bool:
    """L'en-tete zlib est un couple (0x78, drapeau) : deux octets, pas une signature longue."""
    return len(data) > 1 and data[0] == 0x78 and data[1] in (0x01, 0x5E, 0x9C, 0xDA)


def reconnaitre_format(data: bytes) -> dict:
    """Le format du flux, avec sa confiance et la methode qui l'a etabli.

    L'ordre des essais suit le pouvoir discriminant : une signature binaire tranche
    seule ; un JSON valide tranche seul ; un source Python n'est reconnu que sur une
    STRUCTURE (definition, classe, import), pas sur le seul fait qu'il parse --
    `date,region,produit` parse en Python et serait sinon pris pour du code ; la
    forme des lignes decide du CSV, puis du journal.
    """
    riff = _conteneur_riff(data)
    if riff:
        return {"format": riff, "confiance": 1.0, "methode": "signature riff",
                "preuve": data[:12].hex()}
    for signature, nom in SIGNATURES.items():
        if data.startswith(signature):
            return {"format": nom, "confiance": 1.0, "methode": "signature",
                    "preuve": data[:8].hex()}
    if _entete_zlib(data):
        return {"format": "zlib", "confiance": 1.0, "methode": "signature",
                "preuve": data[:4].hex()}
    try:
        texte = data.decode("utf-8")
    except UnicodeDecodeError:
        return {"format": "binaire", "confiance": 0.9, "methode": "non utf-8",
                "preuve": data[:8].hex()}
    try:
        json.loads(texte)
        return {"format": "json", "confiance": 1.0, "methode": "parse json", "preuve": texte[:40]}
    except json.JSONDecodeError:
        pass
    try:
        arbre = ast.parse(texte)
        structure = (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef,
                     ast.Import, ast.ImportFrom)
        if any(isinstance(noeud, structure) for noeud in arbre.body):
            return {"format": "python", "confiance": 0.95, "methode": "ast.parse + structure",
                    "preuve": f"{len(arbre.body)} noeud(s) racine"}
    except SyntaxError:
        pass
    lignes = [ligne for ligne in texte.splitlines() if ligne.strip()]
    titres = [ligne for ligne in lignes if re.match(r"^#{1,6} ", ligne)]
    if titres and len(titres) / len(lignes) >= 0.02:
        return {"format": "markdown", "confiance": 0.85, "methode": "titres en debut de ligne",
                "preuve": f"{len(titres)} titre(s) sur {len(lignes)} ligne(s)"}
    if len(lignes) >= 3:
        for separateur in (",", ";", "\t"):
            arites = [ligne.count(separateur) for ligne in lignes[:64]]
            if min(arites) >= 1 and len(set(arites)) == 1:
                return {"format": "csv", "confiance": 0.9,
                        "methode": f"arite constante ({arites[0] + 1} champs, sep {separateur!r})",
                        "preuve": lignes[0][:40]}
        horodates = [ligne for ligne in lignes
                     if re.match(r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}", ligne)]
        if len(horodates) / len(lignes) >= 0.8:
            return {"format": "journal", "confiance": round(len(horodates) / len(lignes), 3),
                    "methode": "horodatage ISO en debut de ligne", "preuve": lignes[0][:40]}
    return {"format": "texte", "confiance": 0.5, "methode": "utf-8 sans structure reconnue",
            "preuve": texte[:40]}


class ModeleDocument:
    """Le modele d'un document : la donnee structuree, et de quoi la reecrire."""

    def __init__(self, format: str, contenu, options: dict | None = None):
        self.format = format
        self.contenu = contenu
        self.options = options or {}

    def __repr__(self) -> str:
        return f"<modele {self.format} {type(self.contenu).__name__}>"


def analyser_json(data: bytes) -> ModeleDocument:
    texte = data.decode("utf-8")
    return ModeleDocument("json", json.loads(texte), {"final_nl": texte.endswith("\n")})


def serialiser_json(modele: ModeleDocument) -> bytes:
    texte = json.dumps(modele.contenu, indent=2, ensure_ascii=False)
    return (texte + ("\n" if modele.options.get("final_nl") else "")).encode("utf-8")


def analyser_csv(data: bytes) -> ModeleDocument:
    texte = data.decode("utf-8")
    lignes = list(csv.reader(io.StringIO(texte)))
    return ModeleDocument("csv", {"entetes": lignes[0], "lignes": [l for l in lignes[1:] if l]},
                          {"final_nl": texte.endswith("\n"), "sep": ","})


def serialiser_csv(modele: ModeleDocument) -> bytes:
    tampon = io.StringIO()
    graveur = csv.writer(tampon, lineterminator="\n", delimiter=modele.options.get("sep", ","))
    graveur.writerow(modele.contenu["entetes"])
    for ligne in modele.contenu["lignes"]:
        graveur.writerow(ligne)
    texte = tampon.getvalue()
    return texte.encode("utf-8") if modele.options.get("final_nl") else texte[:-1].encode("utf-8")


_MOTIF_JOURNAL = re.compile(r"^(\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}Z) \[([A-Z]+)\] (.*)$")


def analyser_journal(data: bytes) -> ModeleDocument:
    texte = data.decode("utf-8")
    entrees = []
    for ligne in texte.splitlines():
        trouve = _MOTIF_JOURNAL.match(ligne)
        if trouve:
            horodatage, niveau, reste = trouve.groups()
            entrees.append({"horodatage": horodatage, "niveau": niveau, "reste": reste,
                            "champs": dict(re.findall(r"(\w+)=([^\s]+)", reste))})
        else:
            # Une ligne non conforme est gardee VERBATIM : un analyseur qui jette ce
            # qu'il ne comprend pas ne peut pas pretendre reconstruire l'artefact.
            entrees.append({"brut": ligne})
    return ModeleDocument("journal", entrees, {"final_nl": texte.endswith("\n")})


def serialiser_journal(modele: ModeleDocument) -> bytes:
    lignes = [e["brut"] if "brut" in e else f"{e['horodatage']} [{e['niveau']}] {e['reste']}"
              for e in modele.contenu]
    texte = "\n".join(lignes)
    return (texte + ("\n" if modele.options.get("final_nl") else "")).encode("utf-8")


def analyser_python(data: bytes) -> ModeleDocument:
    texte = data.decode("utf-8")
    return ModeleDocument("python", ast.parse(texte), {"final_nl": texte.endswith("\n")})


def serialiser_python(modele: ModeleDocument) -> bytes:
    """`ast.unparse` reecrit la forme : identite SEMANTIQUE, jamais binaire."""
    texte = ast.unparse(modele.contenu)
    return (texte + ("\n" if modele.options.get("final_nl") else "")).encode("utf-8")


def analyser_markdown(data: bytes) -> ModeleDocument:
    texte = data.decode("utf-8")
    titres = [{"niveau": len(m.group(1)), "titre": m.group(2).strip(), "offset": m.start()}
              for m in re.finditer(r"^(#{1,6}) (.+)$", texte, re.M)]
    return ModeleDocument("markdown", {"texte": texte, "titres": titres},
                          {"final_nl": texte.endswith("\n")})


def serialiser_markdown(modele: ModeleDocument) -> bytes:
    # Le modele garde le corps VERBATIM : il n'y a rien a normaliser ici, et
    # pretendre le contraire ferait perdre des octets sans rien gagner.
    return modele.contenu["texte"].encode("utf-8")


ANALYSEURS = {"json": analyser_json, "csv": analyser_csv, "journal": analyser_journal,
              "python": analyser_python, "markdown": analyser_markdown}
SERIALISEURS = {"json": serialiser_json, "csv": serialiser_csv, "journal": serialiser_journal,
                "python": serialiser_python, "markdown": serialiser_markdown}


def premier_ecart(attendu: bytes, obtenu: bytes) -> str:
    """Le premier octet qui differe, situe : un verdict binaire ne dit pas ou ca casse."""
    for i, (a, b) in enumerate(zip(attendu, obtenu)):
        if a != b:
            debut = max(0, i - 8)
            return (f"offset {i} : attendu {attendu[debut:i + 8].hex()} "
                    f"!= obtenu {obtenu[debut:i + 8].hex()}")
    if len(attendu) != len(obtenu):
        return f"prefixe commun, longueurs {len(attendu)} != {len(obtenu)}"
    return "aucun"


def identite_artefact(generateur, artefact: bytes) -> dict:
    """Verifie qu'un code candidat REPRODUIT l'artefact, et nomme l'ecart sinon.

    C'est le garde de la provenance : proposer « le code qui a genere cet artefact »
    n'a de valeur que si le code, rejoue, redonne l'artefact a l'identique.
    """
    propose = generateur()
    return {"identique": propose == artefact,
            "sha_artefact": hashlib.sha256(artefact).hexdigest()[:16],
            "sha_propose": hashlib.sha256(propose).hexdigest()[:16],
            "ecart": "aucun" if propose == artefact else premier_ecart(artefact, propose)}


def identite_analyse(modele: ModeleDocument, data: bytes) -> dict:
    """Les deux identites : binaire (octets) et, pour le source, semantique (arbre)."""
    reecrit = SERIALISEURS[modele.format](modele)
    if reecrit == data:
        return {"binaire": True, "semantique": True, "ecart": "aucun"}
    verdict = {"binaire": False, "ecart": premier_ecart(data, reecrit),
               "octets_origine": len(data), "octets_reecrits": len(reecrit)}
    if modele.format == "python":
        arbre_origine = ast.dump(modele.contenu, include_attributes=False)
        arbre_reecrit = ast.dump(ast.parse(reecrit.decode("utf-8")), include_attributes=False)
        verdict["semantique"] = arbre_origine == arbre_reecrit
    else:
        verdict["semantique"] = False
    return verdict


def _type_valeur(valeur: str) -> str:
    """Le type d'une valeur textuelle : grossier, et declare comme tel par son nom."""
    if re.fullmatch(r"-?\d+", valeur):
        return "entier"
    if re.fullmatch(r"-?\d+[.,]\d+", valeur):
        return "reel"
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}(T[\d:Z+-]+)?", valeur):
        return "date"
    if re.fullmatch(r"(true|false)", valeur, re.I):
        return "booleen"
    return "texte"


def _schema_json(valeur, profondeur: int = 0) -> dict:
    """La forme d'une valeur JSON : type, et pour un objet ses cles, bornees en profondeur."""
    if profondeur >= 2:
        return {"type": type(valeur).__name__}
    if isinstance(valeur, dict):
        return {"type": "objet",
                "cles": {cle: _schema_json(v, profondeur + 1)
                         for cle, v in list(valeur.items())[:12]}}
    if isinstance(valeur, list):
        return {"type": "liste", "longueur": len(valeur),
                "element": _schema_json(valeur[0], profondeur + 1) if valeur else None}
    return {"type": _type_valeur(str(valeur))}


def schema_document(modele: ModeleDocument | None) -> dict:
    """Le SCHEMA du modele du document : ce que le format seul ne dit pas."""
    if modele is None:
        return {}
    if modele.format == "csv":
        colonnes = []
        for j, nom in enumerate(modele.contenu["entetes"]):
            valeurs = [ligne[j] for ligne in modele.contenu["lignes"] if len(ligne) > j]
            colonnes.append({"nom": nom,
                             "types": sorted({_type_valeur(v) for v in valeurs if v != ""}),
                             "nul": any(v == "" for v in valeurs),
                             "valeurs_distinctes": len(set(valeurs))})
        return {"format": "csv", "colonnes": colonnes, "lignes": len(modele.contenu["lignes"])}
    if modele.format == "json":
        return {"format": "json", "structure": _schema_json(modele.contenu)}
    if modele.format == "journal":
        conformes = [e for e in modele.contenu if "brut" not in e]
        champs = Counter(cle for e in conformes for cle in e["champs"])
        return {"format": "journal", "lignes": len(modele.contenu), "conformes": len(conformes),
                "champs": {cle: n for cle, n in champs.most_common(8)},
                "niveaux": dict(Counter(e["niveau"] for e in conformes))}
    if modele.format == "markdown":
        return {"format": "markdown", "titres": modele.contenu["titres"],
                "octets": len(modele.contenu["texte"])}
    if modele.format == "python":
        return {"format": "python",
                "symboles": [{"nom": getattr(noeud, "name", type(noeud).__name__),
                              "genre": type(noeud).__name__,
                              "ligne": getattr(noeud, "lineno", 0)}
                             for noeud in modele.contenu.body]}
    return {}


def reperes_depuis_schema(seg: Segment, reconnu: dict,
                          modele: ModeleDocument | None = None) -> list[Repere]:
    """Les reperes que la STRUCTURE propose -- derives du schema, jamais ecrits en dur."""
    if modele is None:
        analyseur = ANALYSEURS.get(reconnu["format"])
        modele = analyseur(seg.data) if analyseur else None
    if modele is None:
        return []
    if modele.format == "journal":
        # Une frontiere par heure, derivee de la date du PREMIER enregistrement
        # conforme : le motif n'est pas ecrit dans le carnet, il est lu sur le document.
        premier = next((e for e in modele.contenu if "horodatage" in e), None)
        if premier is None:
            return []
        date = premier["horodatage"].split("T")[0]
        return [Repere("ligne_regex", r"^" + date + r"T\d{2}:00:00Z", "heure")]
    if modele.format == "markdown":
        return [Repere("titre_markdown", None, "section")]
    if modele.format == "python":
        return [Repere("ligne_regex", r"^(?:def |class )", "definition")]
    return []


def structure_interne(seg: Segment) -> dict | None:
    """Ce que le FORMAT du segment revele -- reconnu sur les octets, pas sur l'extension."""
    reconnu = reconnaitre_format(seg.data)
    if reconnu["format"] in ("binaire", "texte"):
        return None
    analyseur = ANALYSEURS.get(reconnu["format"])
    if analyseur is None:
        return None
    schema = schema_document(analyseur(seg.data))
    # Resume compact : le schema complet a chaque tour de triage couterait la fenetre.
    if reconnu["format"] == "csv":
        return {"format": "csv", "colonnes": schema["colonnes"], "lignes": schema["lignes"]}
    if reconnu["format"] == "json":
        structure = schema.get("structure", {})
        if structure.get("type") == "liste":
            # Un tableau n'a pas de cles racine : rendre une liste vide laissait
            # croire a un objet muet, et le modele choisissait quand meme une
            # operation par cles. On dit ce que la racine est, et de quoi elle
            # est faite.
            return {"format": "json", "racine": "liste",
                    "longueur": structure.get("longueur"), "element": structure.get("element")}
        return {"format": "json", "cles_racine": list(structure.get("cles", {}))}
    if reconnu["format"] == "journal":
        return {"format": "journal", "champs": list(schema["champs"]), "niveaux": schema["niveaux"]}
    if reconnu["format"] == "markdown":
        return {"format": "markdown", "titres": [t["titre"] for t in schema["titres"]][:8]}
    if reconnu["format"] == "python":
        return {"format": "python", "symboles": [s["nom"] for s in schema["symboles"]][:12]}
    return None


def decoupages_possibles(seg: Segment) -> list[str]:
    """Les decoupages que le carnet sait reellement appliquer a ce segment."""
    # Le menu suit le format RECONNU : renommer le fichier ne change plus le menu.
    if seg.profondeur >= PROFONDEUR_MAX:
        return []
    modes = ["lignes", "morceaux"]
    reconnu = reconnaitre_format(seg.data)
    if reconnu["format"] == "csv":
        modes.append("colonnes_csv")
    if reconnu["format"] == "json":
        # La RACINE decide de l'operation : un objet se decoupe par cles, un
        # tableau par elements. Annoncer "cles_json" sur un tableau promettait
        # une operation que decouper ne sait pas tenir -- une vraie passe l'a
        # choisie sur le champ "clients" de clients.json (une liste de 40
        # objets) et a leve AttributeError sur .items(). Le menu n'annonce
        # plus que ce qu'il sait tenir.
        racine = json.loads(seg.data.decode("utf-8"))
        if isinstance(racine, dict):
            modes.append("cles_json")
        elif isinstance(racine, list):
            modes.append("elements_json")
    if reconnu["format"] == "markdown":
        modes.append("blocs_markdown")
    return modes


def vue(seg: Segment) -> dict:
    """La carte d'un segment telle qu'elle est presentee au modele."""
    # Memes champs que la carte precedente, plus le format reconnu et le schema --
    # c'est ce que le format seul n'apportait pas.
    reconnu = reconnaitre_format(seg.data)
    analyseur = ANALYSEURS.get(reconnu["format"])
    return {
        "id": seg.id,
        "genre": seg.genre,
        "octets": len(seg.data),
        "entropie_bits_octet": round(entropie(seg.data), 3),
        "apercu": apercu(seg.data),
        "format_reconnu": reconnu,
        "schema": schema_document(analyseur(seg.data)) if analyseur else None,
        "structure": structure_interne(seg),
        "decoupages_possibles": decoupages_possibles(seg),
    }


print("carte reecrite sur la reconnaissance d'octets : structure_interne, "
      "decoupages_possibles et vue gardent leur nom et leur signature")
print(json.dumps(vue(SEGMENTS["ventes/ventes.csv#0"]), ensure_ascii=False)[:400])

# L'extension n'est pas la source de la reconnaissance : le meme flux, renomme.
# Ce fichier est cree APRES la moisson de `FICHIERS` : il ne touche aucun total.
_faux = RACINE / "data" / "ventes_sans_extension.dat"
_faux.write_bytes((RACINE / "ventes" / "ventes.csv").read_bytes())
_reconnu_faux = reconnaitre_format(_faux.read_bytes())
assert _reconnu_faux["format"] == "csv", _reconnu_faux
assert sorted(decoupages_possibles(Segment("x#0", _faux.read_bytes(), "data/x.dat")))
print(f"renomme en .dat, le CSV est reconnu : {_reconnu_faux['format']} "
      f"({_reconnu_faux['methode']}, confiance {_reconnu_faux['confiance']})")

# Identite des analyses : binaire (octets) ou semantique (arbre), jamais supposee.
IDENTITES = []
for chemin in FICHIERS:
    relatif = chemin.relative_to(RACINE).as_posix()
    data = chemin.read_bytes()
    reconnu = reconnaitre_format(data)
    analyseur = ANALYSEURS.get(reconnu["format"])
    if analyseur is None:
        IDENTITES.append((relatif, reconnu, None, None))
        continue
    modele = analyseur(data)
    IDENTITES.append((relatif, reconnu, modele, identite_analyse(modele, data)))

print()
print(f"{'fichier':26s} {'format':9s} {'conf':6s} {'identite':11s} methode")
for relatif, reconnu, modele, identite in IDENTITES:
    if identite is None:
        verdict = "sans modele"
    elif identite["binaire"]:
        verdict = "binaire"
    elif identite.get("semantique"):
        verdict = "semantique"
    else:
        verdict = "ROMPUE"
    print(f"  {relatif:26s} {reconnu['format']:9s} {str(reconnu['confiance']):6s} "
          f"{verdict:11s} {reconnu['methode']}")

ecarts = [(r, i) for r, _, _, i in IDENTITES if i is not None and i["ecart"] != "aucun"]
if ecarts:
    print()
    for relatif, identite in ecarts:
        print(f"  ecart {relatif} : {identite['ecart']} "
              f"({identite['octets_origine']} -> {identite['octets_reecrits']} o)")

print()
for relatif, reconnu, modele, identite in IDENTITES:
    if modele is not None:
        print(f"  {relatif:26s} schema : "
              f"{json.dumps(schema_document(modele), ensure_ascii=False)[:170]}")

carte reecrite sur la reconnaissance d'octets : structure_interne, decoupages_possibles et vue gardent leur nom et leur signature
{"id": "ventes/ventes.csv#0", "genre": ".csv", "octets": 4464, "entropie_bits_octet": 4.614, "apercu": "date,region,produit,quantite,prix,canal\\n2026-01-01,nord,vis,", "format_reconnu": {"format": "csv", "confiance": 0.9, "methode": "arite constante (6 champs, sep ',')", "preuve": "date,region,produit,quantite,prix,canal"}, "schema": {"format": "csv", "colonnes": [{"nom": "date", "types": ["date"
renomme en .dat, le CSV est reconnu : csv (arite constante (6 champs, sep ','), confiance 0.9)



fichier                    format    conf   identite    methode
  config/settings.json       json      1.0    binaire     parse json
  data/mesures.bin           binaire   0.9    sans modele non utf-8
  data/mixte.dat             binaire   0.9    sans modele non utf-8
  docs/notes.md              markdown  0.85   binaire     titres en debut de ligne
  logs/app.log               journal   1.0    binaire     horodatage ISO en debut de ligne
  src/analyse.py             python    0.95   semantique  ast.parse + structure
  ventes/clients.json        json      1.0    binaire     parse json
  ventes/ventes.csv          csv       0.9    binaire     arite constante (6 champs, sep ',')

  ecart src/analyse.py : offset 63 : attendu 7569742e2222220a0a66726f6d20636f != obtenu 7569742e2222220a66726f6d20636f6c (1111 -> 1106 o)

  config/settings.json       schema : {"format": "json", "structure": {"type": "objet", "cles": {"seuil": {"type": "reel"}, "reessais": {"type": "entier"}, "regions": {"type

### Lecture du résultat

Deux choses sont mesurées ici, et aucune n'est supposée.

La première est l'**identité** : pour chaque fichier, la resérialisation redonne-t-elle exactement les octets d'origine ? Le verdict est affiché fichier par fichier, et l'écart est *situé* — un offset et les octets fautifs, jamais un simple « différent ». Un format dont le modèle est fidèle se resérialise à l'identique ; un format dont la réécriture normalise la forme ne le peut pas, et le carnet le dit au lieu de le taire — c'est précisément ce qui interdit ensuite de proposer ce sérialiseur comme générateur de l'artefact.

La seconde est l'**indépendance à l'extension** : le même CSV, renommé en `.dat`, est reconnu comme CSV. La reconnaissance porte sur le flux, pas sur le nom.

### Exercice 1 — annoncer la structure d'un format non couvert

`structure_interne` reconnaît `csv`, `json`, `log` et `markdown`. Un format fréquent manque : le fichier de **paires clé=valeur** (`.ini`, `.properties`, `.env`), où chaque ligne non vide et non commentée est une affectation.

**Objectif** : compléter la fonction ci-dessous pour qu'elle rende `{"format": "paires", "cles": [...]}` avec les clés rencontrées dans l'ordre, ou `None` si le texte n'en contient aucune.

**Indice** — une ligne utile vérifie `re.match(r"^[A-Za-z_][A-Za-z0-9_]*\s*=", ligne)`, après avoir retiré les commentaires commençant par `#`.

**Étape 1** : découper le texte en lignes. **Étape 2** : filtrer commentaires et lignes vides. **Étape 3** : extraire la partie gauche du `=`.

La fonction n'est pas branchée sur `structure_interne` : tant qu'elle rend `None`, le carnet se comporte exactement comme avant. Décommentez le branchement en fin de cellule une fois l'exercice fait.

In [7]:
def structure_paires(texte: str):
    """Exercice 1 : reconnaitre un fichier de paires cle=valeur."""
    # TODO etudiant : appliquer les trois etapes de l'enonce.
    return None


# Branchement optionnel, a activer apres l'exercice :
# structure_interne_origine = structure_interne
# def structure_interne(seg):  # noqa: F811
#     paires = structure_paires(seg.data.decode("utf-8", "replace"))
#     return paires if paires else structure_interne_origine(seg)

print(f"Exercice 1 — structure de paires : {'reconnue' if structure_paires('a = 1') else 'a completer'}")

Exercice 1 — structure de paires : a completer


## Les codecs : trois algorithmes complets, et un quatrième à dictionnaire

Un codec est une **primitive** : il transforme un segment en un binaire plus court, et son inverse rend exactement les octets d'origine. Le carnet enregistre quatre codecs, chacun avec son encodeur et son décodeur.

Trois viennent de la bibliothèque standard et sont des algorithmes complets : `zlib` (LZ77 + Huffman), `bz2` (BWT + Huffman), `lzma` (LZ + codage par intervalles). Le quatrième est le codec à dictionnaire construit juste après : c'est celui qui justifie le détour, parce qu'il exploite une redondance que les trois autres ne voient pas — celle qui est **partagée entre segments**.

Le contrat est unique : `CODECS[nom] = (compresser, decompresser)`. Le carnet vérifie ce contrat sur chaque codec avant de le proposer au modèle, sur les octets du corpus eux-mêmes.

In [8]:
def _identite(data: bytes) -> bytes:
    return data


CODECS: dict[str, tuple] = {
    "zlib": (lambda d: zlib.compress(d, 9), zlib.decompress),
    "bz2": (lambda d: bz2.compress(d, 9), bz2.decompress),
    "lzma": (lambda d: lzma.compress(d, preset=9 | lzma.PRESET_EXTREME), lzma.decompress),
    "store": (_identite, _identite),
}

# Verification du contrat : tout codec enregistre doit rendre les octets d'origine,
# sur du texte, sur du binaire et sur du vide. Un codec non inversible ne doit
# jamais atteindre le modele.
temoin_texte = (RACINE / "ventes" / "ventes.csv").read_bytes()
temoin_binaire = (RACINE / "data" / "mesures.bin").read_bytes()[:4000]
for nom, (compresser, decompresser) in CODECS.items():
    for temoin in (b"", temoin_texte[:600], temoin_binaire):
        assert decompresser(compresser(temoin)) == temoin, f"codec non inversible : {nom}"
print(f"codecs enregistres et inversibles verifies : {sorted(CODECS)}")

codecs enregistres et inversibles verifies : ['bz2', 'lzma', 'store', 'zlib']


### Lecture du résultat

Chaque codec est passé sur du vide, sur du texte et sur du binaire **avant** d'être présenté au modèle. Ce n'est pas une formalité : un codec dont l'inverse perd un octet produirait une décompression qui « marche » sur les cas faciles et corrompt un fichier sur les autres, et le carnet ne le verrait qu'à la comparaison finale — trop tard pour savoir lequel.

La vérification est un `assert`, donc elle interrompt le carnet. C'est voulu : un codec non inversible est une erreur de programmation, pas une donnée à rapporter.

### Le codec à dictionnaire

Les trois codecs de la bibliothèque standard repartent de zéro sur **chaque segment**. Or les segments d'un même corpus se ressemblent : une colonne de catégories répète les mêmes mots, un journal répète les mêmes niveaux, un JSON répète les mêmes clés.

Un **codec à dictionnaire** apprend une fois les sous-chaînes fréquentes du corpus, transmet ce dictionnaire **une seule fois**, puis remplace chaque occurrence par un index compact. C'est le principe des dictionnaires de `zstd` ou de `brotli` : la redondance inter-segments devient du gain sur chaque segment pris séparément.

Le format du flux produit est simple et sans ambiguïté :

| Octet émis | Sens |
|---|---|
| `0x00 0x00` | l'octet littéral `0x00` |
| `0x00 (i+1)` | l'entrée `i` du dictionnaire (`i` de 0 à 254) |
| tout autre octet | lui-même |

Le décodeur lit donc sans jamais hésiter : après un `0x00`, le second octet tranche entre littéral et référence. Le dictionnaire appris voyage dans un en-tête compressé, dont la taille est **comptée** dans le total.

In [9]:
MARQUEUR = 0x00
TAILLE_DICO_MAX = 255


def apprendre_dictionnaire(donnees: bytes, longueurs=(16, 12, 8), max_entrees=96) -> list[bytes]:
    """Choisit des sous-chaines frequentes, par gain brut (occurrences - 1) * longueur."""
    echantillon = donnees[:16384]
    frequences: Counter = Counter()
    for longueur in longueurs:
        for i in range(len(echantillon) - longueur + 1):
            frequences[echantillon[i:i + longueur]] += 1

    retenus: list[bytes] = []
    occupes: set[bytes] = set()
    for motif, compte in frequences.most_common(4000):
        if len(retenus) >= max_entrees:
            break
        # Gain : on remplace `compte` occurrences par 2 octets chacune.
        if compte < 2 or (compte - 1) * (len(motif) - 2) < len(motif) + 4:
            continue
        # Pas de motif inclus dans un motif deja retenu : il ne serait jamais atteint.
        if any(motif in deja or deja in motif for deja in occupes):
            continue
        retenus.append(motif)
        occupes.add(motif)
    return retenus[:TAILLE_DICO_MAX]


def encoder_dictionnaire(donnees: bytes, dico: list[bytes]) -> bytes:
    index = {motif: i for i, motif in enumerate(dico)}
    sortie = bytearray()
    i = 0
    while i < len(donnees):
        meilleur = None
        for motif in dico:
            if donnees.startswith(motif, i) and (meilleur is None or len(motif) > len(meilleur)):
                meilleur = motif
        if meilleur is not None:
            sortie.append(MARQUEUR)
            sortie.append(index[meilleur] + 1)
            i += len(meilleur)
        else:
            octet = donnees[i]
            sortie.append(octet)
            if octet == MARQUEUR:
                sortie.append(MARQUEUR)
            i += 1
    return bytes(sortie)


def decoder_dictionnaire(donnees: bytes, dico: list[bytes]) -> bytes:
    sortie = bytearray()
    i = 0
    while i < len(donnees):
        octet = donnees[i]
        if octet != MARQUEUR:
            sortie.append(octet)
            i += 1
            continue
        suivant = donnees[i + 1]
        if suivant == 0:
            sortie.append(0)
        else:
            sortie += dico[suivant - 1]
        i += 2
    return bytes(sortie)


# Le dictionnaire est appris UNE fois sur le corpus, puis transmis une seule fois.
DICO = apprendre_dictionnaire(b"".join(p.read_bytes()[:4096] for p in FICHIERS))
EN_TETE_DICO = zlib.compress(
    json.dumps([m.decode("latin-1") for m in DICO]).encode("utf-8"), 9
)
DICO_DECODE = [m.encode("latin-1") for m in json.loads(zlib.decompress(EN_TETE_DICO))]


def _compresser_dico(donnees: bytes) -> bytes:
    return len(EN_TETE_DICO).to_bytes(4, "big") + EN_TETE_DICO + zlib.compress(
        encoder_dictionnaire(donnees, DICO_DECODE), 9
    )


def _decompresser_dico(donnees: bytes) -> bytes:
    taille = int.from_bytes(donnees[:4], "big")
    dico = [m.encode("latin-1") for m in json.loads(zlib.decompress(donnees[4:4 + taille]))]
    return decoder_dictionnaire(zlib.decompress(donnees[4 + taille:]), dico)


CODECS["dict"] = (_compresser_dico, _decompresser_dico)

for temoin in (b"", temoin_texte[:600], temoin_binaire):
    assert _decompresser_dico(_compresser_dico(temoin)) == temoin, "codec dict non inversible"

print(f"dictionnaire appris : {len(DICO)} entrees, {len(EN_TETE_DICO)} o une fois compresse")
print(f"exemples            : {[m.decode('latin-1')[:28] for m in DICO[:6]]}")
print(f"codecs proposes au modele : {sorted(CODECS)}")

dictionnaire appris : 96 entrees, 403 o une fois compresse
exemples            : ['ligne de mesure ', 'igne de mesure r', 'gne de mesure re', 'ne de mesure rep', 'e de mesure repe', ' de mesure repet']
codecs proposes au modele : ['bz2', 'dict', 'lzma', 'store', 'zlib']


### Lecture du résultat

Le dictionnaire a retenu des motifs du corpus — des fragments de lignes CSV, de niveaux de journal, de clés JSON. Il est appris **une fois** et transmis **une fois** : son en-tête compressé est payé au total une seule fois, pas à chaque segment. C'est exactement la structure de coût d'un dictionnaire de `zstd`, et c'est ce qui rend le quatrième codec honnêtement comparable aux trois autres.

Le gain n'est pas garanti sur un segment isolé : un segment de 40 Ko d'octets aléatoires ne contient aucun motif du dictionnaire, donc le codec y ajoute un `0x00` échappé de temps en temps sans rien retirer. C'est au modèle de ne pas le choisir là — et au carnet de publier le résultat même si le modèle se trompe.

### Exercice 2 — un cinquième codec, inversible par construction

Le codec le plus simple qui existe est le **RLE** (*run-length encoding*) : une suite de `n` fois le même octet devient le couple `(octet, n)`. Il est excellent sur les images de plats de couleur unie, catastrophique sur du texte, et surtout il est **trivialement inversible** — ce qui en fait un bon terrain d'exercice.

**Objectif** : compléter `compresser_rle` et `decompresser_rle` de sorte que `decompresser_rle(compresser_rle(x)) == x` pour tout `x`.

**Indice** — un encodage sans ambiguïté : émettre `octet` puis `nombre` sur **deux octets** à chaque fois (donc un maximum de 255 répétitions par groupe). Le décodeur lit la sortie par paires, sans jamais devoir deviner.

**Étape 1** : parcourir l'entrée en comptant les répétitions consécutives. **Étape 2** : à chaque changement, émettre le couple. **Étape 3** : ne pas oublier la dernière série.

**Attention** : ce codec ne doit pas être ajouté à `CODECS` avant d'être correct — sinon le modèle pourrait le choisir pendant le triage et le carnet échouerait au milieu de la passe. La cellule de vérification ne s'exécute que si l'exercice rend quelque chose.

In [10]:
def compresser_rle(donnees: bytes):
    """Exercice 2 : encoder les repetitions consecutives en couples (octet, nombre)."""
    # TODO etudiant : suivre les trois etapes de l'enonce.
    return None


def decompresser_rle(donnees: bytes):
    """Exercice 2 : reconstruire les octets d'origine."""
    # TODO etudiant : lire l'entree par paires.
    return None


_essai = compresser_rle(b"aaabbbbbcc")
if _essai is None:
    print("Exercice 2 a completer : codec RLE")
else:
    for _temoin in (b"", b"a", b"aaa", b"ababab", temoin_binaire):
        assert decompresser_rle(compresser_rle(_temoin)) == _temoin, "RLE non inversible"
    CODECS["rle"] = (compresser_rle, decompresser_rle)
    print(f"RLE injecte et verifie ; codecs : {sorted(CODECS)}")

Exercice 2 a completer : codec RLE


## La recette, et l'invariant de couverture

Une opération est **inversible** et **déclarée**. Le carnet en connaît deux familles :

| Famille | Opération | Effet | Inverse |
|---|---|---|---|
| primitive | `compresser(segment, codec)` | segment → segment compressé en binaire | le codec correspondant |
| intermédiaire | `decouper(segment, mode)` | segment macro → plusieurs segments qualifiés | le recollage propre au mode |

La **recette** est la liste ordonnée des décisions. Elle ne contient aucune donnée du corpus : uniquement des identifiants de segments, des noms de codecs, des noms de modes et, pour les découpages non contigus — colonnes d'un CSV, clés d'un JSON — la colle minimale qui permet de recoller.

**L'invariant de couverture est le cœur du contrat** : un segment original est *couvert* s'il a reçu un traitement final, ou s'il a été découpé en enfants tous couverts. Le calcul est récursif, et il **lève** une exception en nommant les segments oubliés. Une passe qui laisse un segment sans traitement n'est pas une passe incomplète à signaler : c'est une passe échouée.

In [11]:
class CouvertureIncomplete(RuntimeError):
    """Leve quand un segment original n'a recu aucun traitement final."""


def recoller(mode: str, enfants: list[bytes], colle: dict) -> bytes:
    """L'inverse d'un decoupage : reconstruit les octets du segment parent."""
    if mode in ("lignes", "morceaux", "blocs_markdown", "reperes"):
        return b"".join(enfants)
    if mode == "colonnes_csv":
        colonnes = [e.decode("utf-8").split("\n") for e in enfants]
        n = len(colonnes[0])
        lignes = [colle["sep"].join(col[r] for col in colonnes) for r in range(n)]
        return (colle["nl"].join(lignes) + (colle["nl"] if colle["final_nl"] else "")).encode("utf-8")
    if mode == "cles_json":
        objet = {cle: json.loads(e.decode("utf-8")) for cle, e in zip(colle["cles"], enfants)}
        return (json.dumps(objet, indent=2, ensure_ascii=False)
                + ("\n" if colle["final_nl"] else "")).encode("utf-8")
    if mode == "elements_json":
        liste = [json.loads(e.decode("utf-8")) for e in enfants]
        return (json.dumps(liste, indent=2, ensure_ascii=False)
                + ("\n" if colle["final_nl"] else "")).encode("utf-8")
    raise ValueError(f"mode de recolle inconnu : {mode}")


def decouper(seg: Segment, mode: str, prefixe: str) -> tuple[list[Segment], dict]:
    """Operation intermediaire : un segment macro -> des segments qualifies + la colle."""
    if mode not in decoupages_possibles(seg):
        raise ValueError(f"{mode} ne s'applique pas a {seg.id}")
    profondeur = seg.profondeur + 1

    if mode in ("lignes", "morceaux", "blocs_markdown", "reperes"):
        if mode == "lignes":
            # keepends=True : chaque groupe garde ses propres retours a la ligne,
            # donc la concatenation b"".join des groupes est exacte. Un split
            # naif par \n perd les joints aux frontieres de groupes (mesure :
            # une vraie passe a reconstruit app.log ampute de 3 octets).
            lignes = seg.data.splitlines(keepends=True)
            parts = max(1, min(4, len(lignes) // 20))
            taille = max(1, len(lignes) // parts)
            morceaux = [b"".join(lignes[i:i + taille]) for i in range(0, len(lignes), taille)]
        elif mode == "morceaux":
            parts = 2
            taille = max(1, len(seg.data) // parts)
            morceaux = [seg.data[i:i + taille] for i in range(0, len(seg.data), taille)]
        else:
            texte = seg.data.decode("utf-8")
            positions = [m.start() for m in re.finditer(r"^##? ", texte, re.M)]
            bornes = sorted({0, *positions, len(texte)})
            morceaux = [texte[a:b].encode("utf-8") for a, b in zip(bornes, bornes[1:])]
        enfants = [Segment(f"{prefixe}.{i}", m, f"{seg.genre}/{mode}", seg.id, profondeur)
                   for i, m in enumerate(morceaux) if m]
        return enfants, {"mode": mode}

    if mode == "colonnes_csv":
        texte = seg.data.decode("utf-8")
        final_nl = texte.endswith("\n")
        lignes = (texte[:-1] if final_nl else texte).split("\n")
        cellules = [ligne.split(",") for ligne in lignes]
        enfants = []
        for j in range(len(cellules[0])):
            colonne = "\n".join(rangee[j] for rangee in cellules).encode("utf-8")
            enfants.append(Segment(f"{prefixe}.c{j}", colonne,
                                   f"{seg.genre}/colonne:{cellules[0][j]}", seg.id, profondeur))
        return enfants, {"mode": mode, "sep": ",", "nl": "\n", "final_nl": final_nl}

    if mode == "cles_json":
        texte = seg.data.decode("utf-8")
        final_nl = texte.endswith("\n")
        objet = json.loads(texte)
        if not isinstance(objet, dict):
            # Garde de contrat : meme si un appelant force le mode, l'erreur
            # nomme le segment et la racine au lieu de partir en AttributeError.
            raise ValueError(f"cles_json demande un objet JSON a la racine, "
                             f"{seg.id} porte un {type(objet).__name__}")
        enfants = [Segment(f"{prefixe}.k{i}",
                           json.dumps(valeur, indent=2, ensure_ascii=False).encode("utf-8"),
                           f"{seg.genre}/cle:{cle}", seg.id, profondeur)
                   for i, (cle, valeur) in enumerate(objet.items())]
        return enfants, {"mode": mode, "cles": list(objet), "final_nl": final_nl}

    if mode == "elements_json":
        texte = seg.data.decode("utf-8")
        final_nl = texte.endswith("\n")
        liste = json.loads(texte)
        if not isinstance(liste, list):
            raise ValueError(f"elements_json demande un tableau JSON a la racine, "
                             f"{seg.id} porte un {type(liste).__name__}")
        enfants = [Segment(f"{prefixe}.e{i}",
                           json.dumps(valeur, indent=2, ensure_ascii=False).encode("utf-8"),
                           f"{seg.genre}/element:{i}", seg.id, profondeur)
                   for i, valeur in enumerate(liste)]
        return enfants, {"mode": mode, "n": len(liste), "final_nl": final_nl}

    raise ValueError(f"mode inconnu : {mode}")


# Contrat d'inversibilite des decoupages, symetrique de celui des codecs :
# decouper puis recoller doit redonner les octets d'origine, pour chaque mode.
# Les temoins couvrent le cas limite qui a fait echouer une vraie passe avant ce
# garde-fou : un texte de plus de 40 lignes dont la derniere se termine par un
# retour a la ligne, decoupe en plusieurs groupes -- c'est au joint entre deux
# groupes qu'un split naif perd ses octets de joint.
_lignes_avec_final = "".join(f"ligne-{i:02d}\n" for i in range(45)).encode("utf-8")
_lignes_sans_final = "\n".join(f"ligne-{i:02d}" for i in range(45)).encode("utf-8")
_paires_decoupage = [
    ("lignes", _lignes_avec_final, "texte/journal.log"),
    ("lignes", _lignes_sans_final, "texte/journal.log"),
    ("morceaux", bytes(range(256)) * 3, "bin/brut.bin"),
    ("blocs_markdown", b"# Titre\n\ntexte\n\n## Sous-titre\n\ntexte\n", "texte/notes.md"),
    ("colonnes_csv", b"alpha,beta\n1,2\n3,4\n", "donnees/ventes.csv"),
    ("cles_json", (json.dumps({"a": 1, "b": [1, 2]}, indent=2, ensure_ascii=False)
                   + "\n").encode("utf-8"), "donnees/clients.json"),
    ("elements_json", (json.dumps([{"a": 1}, {"b": [2, 3]}], indent=2, ensure_ascii=False)
                       + "\n").encode("utf-8"), "donnees/clients.json"),
]
for _mode, _donnees, _genre in _paires_decoupage:
    _temoin = Segment(f"temoin/{_genre}", _donnees, _genre)
    assert _mode in decoupages_possibles(_temoin), f"temoin invalide pour {_mode}"
    _enfants, _colle = decouper(_temoin, _mode, "temoin")
    assert _enfants, f"decoupage vide pour {_mode}"
    assert recoller(_mode, [e.data for e in _enfants], _colle) == _donnees, \
        f"decoupage non inversible : {_mode}"
print("decoupages inversibles verifies : lignes (avec et sans retour final), "
      "morceaux, blocs_markdown, colonnes_csv, cles_json, elements_json")


def _couverture(sid: str, recette: dict, vus: set) -> tuple[bool, list[str]]:
    """Un segment est couvert s'il est compresse, ou decoupe en enfants tous couverts."""
    if sid in vus:
        raise CouvertureIncomplete(f"cycle dans la recette autour de {sid}")
    vus = vus | {sid}
    entree = recette.get(sid)
    if entree is None:
        return False, [sid]
    if entree["op"] == "compresser":
        return True, []
    manquants: list[str] = []
    for enfant in entree["enfants"]:
        ok, absents = _couverture(enfant, recette, vus)
        if not ok:
            manquants.extend(absents)
    return not manquants, manquants


def verifier_couverture(racines: list[str], recette: dict) -> None:
    """Invariant : leve en nommant tout segment original non traite."""
    manquants: list[str] = []
    for sid in racines:
        ok, absents = _couverture(sid, recette, set())
        if not ok:
            manquants.extend(absents)
    if manquants:
        raise CouvertureIncomplete(
            f"{len(manquants)} segment(s) sans traitement final : {manquants[:6]}"
        )


print(f"{len(decoupages_possibles(SEGMENTS['ventes/ventes.csv#0']))} decoupages possibles pour le CSV")
print(f"exemple : {decoupages_possibles(SEGMENTS['data/mesures.bin#0'])} pour le binaire")

decoupages inversibles verifies : lignes (avec et sans retour final), morceaux, blocs_markdown, colonnes_csv, cles_json, elements_json
3 decoupages possibles pour le CSV
exemple : ['lignes', 'morceaux'] pour le binaire


### Lecture du résultat

Le CSV accepte `colonnes_csv` ; le binaire ne l'accepte pas, et ne propose même pas de découpage markdown ou JSON. Cette asymétrie est calculée à partir du type du segment, pas déclarée à la main : c'est elle qui empêche le modèle de proposer une opération inapplicable.

Le binaire conserve `lignes` et `morceaux` — découper des octets reste possible même sans structure connue, simplement cela n'apporte rien. Le modèle devra le découvrir, ou le carnet le lui montrera par la mesure.

Le contrat vaut à l'intérieur d'un document, pas seulement à sa racine. `clients.json` est un objet, mais sa clé `clients` porte un tableau : recollée en JSON, la valeur de cette clé a une **racine liste**, donc pas de clés. Le menu le dit — `cles_json` pour un objet, `elements_json` pour un tableau — et un mode qui ne correspond pas à la racine rencontrée est refusé explicitement, en nommant le segment et le type qu'il porte. Un menu qui promet une opération que le découpage ne sait pas tenir déplace l'erreur loin de l'endroit où elle est née.

### L'invariant doit échouer quand il le faut

Un invariant qu'on n'a jamais vu échouer n'est pas un invariant, c'est une fonction qui rend `True`. La cellule suivante construit **exprès** une recette incomplète — un CSV découpé en colonnes dont une seule est compressée — et vérifie que `verifier_couverture` lève en **nommant** les colonnes oubliées.

C'est le contrôle du contrôle : sans lui, l'invariant de la passe réelle pourrait passer pour de mauvaises raisons.

In [12]:
temoin_id = "ventes/ventes.csv#0"
temoin_enfants, temoin_colle = decouper(SEGMENTS[temoin_id], "colonnes_csv", "temoin")

recette_incomplete = {
    temoin_id: {"op": "decouper", "mode": "colonnes_csv", "colle": temoin_colle,
                "enfants": [e.id for e in temoin_enfants]},
}
recette_incomplete[temoin_enfants[0].id] = {"op": "compresser", "codec": "zlib"}

try:
    verifier_couverture([temoin_id], recette_incomplete)
    raise AssertionError("l'invariant n'a pas vu les colonnes non traitees")
except CouvertureIncomplete as erreur:
    print(f"invariant declenche comme attendu : {erreur}")

# Controle positif : la meme recette, toutes colonnes traitees, doit passer.
for enfant in temoin_enfants:
    recette_incomplete[enfant.id] = {"op": "compresser", "codec": "zlib"}
verifier_couverture([temoin_id], recette_incomplete)
print("controle positif : recette complete acceptee")

invariant declenche comme attendu : 5 segment(s) sans traitement final : ['temoin.c1', 'temoin.c2', 'temoin.c3', 'temoin.c4', 'temoin.c5']
controle positif : recette complete acceptee


### Lecture du résultat

L'invariant a nommé les colonnes oubliées, puis a accepté la même recette une fois toutes les colonnes traitées. Le contrôle positif compte autant que le négatif : un invariant qui refuserait **tout** serait aussi inutile qu'un invariant qui accepte tout, et le carnet ne saurait pas distinguer les deux sans l'avoir éprouvé dans les deux sens.

## Le harnais Semantic Kernel

Le carnet 07 écrivait ses schémas d'outils à la main en JSON et postait ses requêtes avec `urllib`. Le dépôt documente déjà le harnais qui fait cela proprement — `Kernel`, plugins `@kernel_function`, `FunctionChoiceBehavior.Auto()` — dans la série GenAI (voir `GenAI/Texte/18_Semantic_Kernel_Plugins.ipynb` et le cas d'étude Medical-Chatbot). Ce carnet **réutilise** ce harnais plutôt que d'en écrire un second.

Le plugin expose deux fonctions, exactement les deux familles d'opérations. Le modèle les voit comme des outils ; quand il en choisit une, **la fonction Python s'exécute en processus** et son résultat lui est renvoyé. Ce n'est pas une simulation : le compteur d'appels et la trace des décisions plus bas sont la preuve que la fonction a réellement tourné.

Un piège mesuré sur cette pile : le modèle servi est un modèle de **raisonnement**, et il consomme des jetons de raisonnement avant de choisir. Avec `max_tokens` trop bas, il ne rend ni texte ni appel d'outil — la réponse est vide. La cellule fixe donc un budget large ; c'est un fait mesuré, pas une marge de confort.

Un second comportement, mesuré lors de la première exécution : sur le segment d'octets pseudo-aléatoires, le modèle a répondu en prose — jugeant, à raison, que rien n'y est compressible — **sans appeler la fonction**. `demander_decision` traite ce cas par une **relance unique** : elle cite la réponse en texte libre au modèle et redemande l'appel d'outil. La relance est un appel réel, compté dans `COMPTEUR` ; si le modèle persiste, la passe devient rouge — elle ne choisit pas à sa place.

In [13]:
class PluginSegments:
    """Plugin Semantic Kernel : les deux operations inversibles, exposees au modele.

    Le plugin ne connait que le segment courant. Il valide l'operation demandee
    et, si elle ne s'applique pas, renvoie la raison au modele plutot que de
    lever : le modele peut alors se corriger dans le meme tour.
    """

    def __init__(self, etat: dict):
        self.etat = etat

    @kernel_function(
        name="compresser",
        description=(
            "Applique un traitement FINAL a un segment : il devient un binaire compresse "
            "et ne sera plus jamais decoupe. Codecs disponibles : zlib, bz2, lzma, dict, store."
        ),
    )
    def compresser(self, codec: str) -> str:
        seg = self.etat["segment"]
        if codec not in CODECS:
            return f"codec inconnu {codec!r} ; disponibles : {sorted(CODECS)}"
        self.etat["decision"] = {"op": "compresser", "codec": codec}
        return f"accepte : {seg.id} compresse avec {codec}"

    @kernel_function(
        name="decouper",
        description=(
            "Separe un segment macro en sous-segments qualifies, qui recevront chacun "
            "un traitement ulterieur. Utiliser les modes listes dans `decoupages_possibles`."
        ),
    )
    def decouper(self, mode: str) -> str:
        seg = self.etat["segment"]
        possibles = decoupages_possibles(seg)
        if mode not in possibles:
            return f"mode {mode!r} inapplicable a {seg.id} ; possibles : {possibles}"
        self.etat["decision"] = {"op": "decouper", "mode": mode}
        return f"accepte : {seg.id} decoupe en {mode}"


async def demander_decision(seg: Segment, carte: str) -> dict | None:
    """Un aller-retour avec le modele pour un segment. Compte l'appel reel."""
    global DERNIER_TEXTE_LIBRE
    etat = {"segment": seg, "decision": None}
    kernel = kernel_pour_decision(etat)

    reglages = OpenAIPromptExecutionSettings(
        max_tokens=3000,
        temperature=0.0,
        function_choice_behavior=FunctionChoiceBehavior.Auto(
            auto_invoke=True, maximum_auto_invoke_attempts=2
        ),
    )
    historique = ChatHistory()
    historique.add_system_message(
        "Tu es un moteur de triage de compression. Pour le segment courant, tu appelles "
        "EXACTEMENT UNE des deux fonctions du plugin `segments` : soit `compresser` (traitement "
        "final, irreversible), soit `decouper` (separation en sous-segments, chacun traite "
        "ensuite). Tu ne reponds jamais en texte libre : tu appelles une fonction, puis tu "
        "conclus en une phrase."
    )
    historique.add_user_message(
        "Carte du corpus (segments deja traites omis) :\n"
        f"{carte}\n\n"
        f"Segment a traiter maintenant : {json.dumps(vue(seg), ensure_ascii=False)}\n\n"
        f"Codecs : {sorted(CODECS)}. Modes applicables : {decoupages_possibles(seg)}.\n"
        "Choisis : `decouper` si la structure interne du segment permet d'isoler des parties "
        "de nature differente (colonnes, cles, blocs) ; sinon `compresser` avec le codec le "
        "plus adapte a ce contenu. Un segment binaire sans structure doit etre compresse, "
        "pas decoupe."
    )

    service = kernel.get_service(SERVICE)
    COMPTEUR["appels"] += 1
    reponse = await service.get_chat_message_contents(
        chat_history=historique, settings=reglages, kernel=kernel
    )

    if etat["decision"] is None:
        # Le modele a repondu en texte libre, ou n'a rien produit. Une relance
        # unique cite sa reponse et redemande l'appel d'outil : c'est un vrai
        # tour d'agent, compte comme tel dans COMPTEUR.
        texte_libre = str(reponse[0]).strip()[:200] or "(reponse vide)"
        DERNIER_TEXTE_LIBRE = texte_libre
        historique.add_assistant_message(texte_libre)
        historique.add_user_message(
            "Tu viens de repondre en texte libre au lieu d'appeler une fonction. "
            "Appelle maintenant exactement une fonction du plugin : "
            f"compresser(codec=...) ou decouper(mode=...). "
            f"Modes applicables ici : {decoupages_possibles(seg)}."
        )
        reglages.max_tokens = 4000
        COMPTEUR["appels"] += 1
        await service.get_chat_message_contents(
            chat_history=historique, settings=reglages, kernel=kernel
        )
    return etat["decision"]


aoai = AsyncOpenAI(api_key=CLE, base_url=BASE)
DERNIER_TEXTE_LIBRE = ""


def kernel_pour_decision(etat: dict) -> Kernel:
    """Un kernel NEUF par decision, portant exactement un plugin.

    Mesure a l'appui : avec un kernel partage, chaque appel y ajoute une paire
    compresser/decouper de plus ; des le deuxieme segment, le modele voit
    quatre outils dont une paire liee a l'etat du segment PRECEDENT, et il
    refuse d'en appeler aucun (deux executions reelles, meme refus au meme
    segment). Reconstruire le kernel par decision elimine la classe entiere.
    """
    k = Kernel()
    k.add_service(OpenAIChatCompletion(ai_model_id=MODELE, async_client=aoai, service_id=SERVICE))
    k.add_plugin(PluginSegments(etat), plugin_name="segments")
    return k


print(f"kernel pret : service {SERVICE!r} -> {MODELE} (un kernel neuf par decision)")

kernel pret : service 'vllm-local' -> qwen3.6-35b-a3b (un kernel neuf par decision)


### Lecture du résultat

Le kernel est construit sur le client OpenAI asynchrone pointé sur l'endpoint local — c'est la forme déjà employée par la série GenAI, à ceci près que l'endpoint est ici la pile vLLM de la maison plutôt qu'un fournisseur externe.

Deux détails du harnais méritent d'être lus. D'abord, **un kernel neuf par décision** : les plugins s'ajoutent à un kernel et ne s'en retirent pas, et nous l'avons mesuré — avec un kernel partagé, au deuxième segment le modèle voyait quatre fonctions, dont une paire `compresser`/`decouper` encore liée à l'état du segment précédent, et il refusait d'en appeler aucune (deux exécutions réelles, même refus au même segment). Reconstruire le kernel à chaque décision élimine la classe d'erreur entière. Ensuite, une opération inapplicable **ne lève pas** : elle rend un message que le modèle lit dans le même tour. C'est ce qui permet à `maximum_auto_invoke_attempts=2` d'avoir un sens — la seconde tentative du modèle dispose de la raison de l'échec de la première.

## Le triage : une boucle jusqu'à couverture complète

Chaque tour traite **un** segment en attente : le modèle reçoit la carte du corpus, la vue du segment, et choisit. Le carnet applique la décision, met à jour la file, et recommence jusqu'à ce qu'il ne reste plus rien.

Deux bornes protègent la boucle, et elles ont des rôles différents :

- **la profondeur** (`PROFONDEUR_MAX = 2`) est structurelle : elle empêche un découpage infini ;
- **le nombre d'appels** (`BORNE_APPELS`) est un garde-fou de coût. Si la boucle l'atteint alors qu'il reste des segments, la passe **échoue** — ce n'est pas une fin normale, c'est un triage qui n'a pas abouti.

La couverture est vérifiée **après** la boucle, par l'invariant, et non supposée parce que la file est vide. Une file vide et une couverture complète sont deux choses différentes : la seconde se prouve.

In [14]:
BORNE_APPELS = 64


async def trier(segments: list[Segment]) -> dict:
    """Boucle de triage : le modele est rappele tant qu'un segment n'a pas de traitement final."""
    recette: dict = {}
    a_traiter = list(segments)
    tous: dict[str, Segment] = {s.id: s for s in segments}

    while a_traiter:
        if COMPTEUR["appels"] >= BORNE_APPELS:
            raise RuntimeError(
                f"triage interrompu a {COMPTEUR['appels']} appels avec "
                f"{len(a_traiter)} segment(s) non traites — la passe echoue."
            )
        seg = a_traiter.pop(0)
        carte = json.dumps(
            [{"id": s.id, "genre": s.genre, "octets": len(s.data),
              "structure": structure_interne(s)} for s in a_traiter[:24]],
            ensure_ascii=False,
        )
        decision = await demander_decision(seg, carte)
        if decision is None:
            detail = f" ; dernier texte libre du modele : {DERNIER_TEXTE_LIBRE[:120]!r}" if DERNIER_TEXTE_LIBRE else ""
            raise RuntimeError(f"le modele n'a choisi aucune operation pour {seg.id} — passe rouge{detail}")

        if decision["op"] == "compresser":
            compresse, _ = CODECS[decision["codec"]]
            seg.data = compresse(seg.data)
            seg.genre = f"{seg.genre}/compresse:{decision['codec']}"
            recette[seg.id] = {"op": "compresser", "codec": decision["codec"]}
        else:
            enfants, colle = decouper(seg, decision["mode"], seg.id.rsplit("#", 1)[0])
            recette[seg.id] = {"op": "decouper", "mode": decision["mode"],
                               "colle": colle, "enfants": [e.id for e in enfants]}
            for enfant in enfants:
                tous[enfant.id] = enfant
            a_traiter.extend(enfants)
        # Une ligne par decision, affichee AU FUR ET A MESURE : chaque appel au
        # modele est un aller-retour de plusieurs secondes, et la passe complete
        # en compte une vingtaine -- sans ce flush, la cellule resterait muette
        # (et le fichier de sortie avec) pendant toute la passe.
        print(f"[appel {COMPTEUR['appels']:3d}] {seg.id:32s} -> "
              f"{decision['op']:10s} {decision.get('codec') or decision.get('mode')}",
              flush=True)

    verifier_couverture([s.id for s in segments], recette)
    return {"recette": recette, "segments": tous}


resultat = await trier(list(SEGMENTS.values()))
recette = resultat["recette"]
print(f"appels LLM reellement effectues : {COMPTEUR['appels']}")
print(f"segments dans l'arbre final     : {len(resultat['segments'])}")
print()
for sid, entree in recette.items():
    if entree["op"] == "compresser":
        print(f"  {sid:44s} compresser  {entree['codec']}")
    else:
        print(f"  {sid:44s} decouper    {entree['mode']} -> {len(entree['enfants'])} segments")

[appel   1] config/settings.json#0           -> compresser zlib


[appel   2] data/mesures.bin#0               -> compresser store


[appel   3] data/mixte.dat#0                 -> compresser lzma


[appel   4] docs/notes.md#0                  -> compresser lzma


[appel   5] logs/app.log#0                   -> compresser lzma


[appel   6] src/analyse.py#0                 -> decouper   morceaux


... (10 outputs left out)

[appel  17] ventes/ventes.csv.c3             -> compresser dict


[appel  18] ventes/ventes.csv.c4             -> compresser lzma


[appel  19] ventes/ventes.csv.c5             -> compresser dict


appels LLM reellement effectues : 19
segments dans l'arbre final     : 19

  config/settings.json#0                       compresser  zlib
  data/mesures.bin#0                           compresser  store
  data/mixte.dat#0                             compresser  lzma
  docs/notes.md#0                              compresser  lzma
  logs/app.log#0                               compresser  lzma
  src/analyse.py#0                             decouper    morceaux -> 3 segments
  ventes/clients.json#0                        decouper    cles_json -> 2 segments
  ventes/ventes.csv#0                          decouper    colonnes_csv -> 6 segments
  src/analyse.py.0                             compresser  zlib
  src/analyse.py.1                             compresser  zlib
  src/analyse.py.2                             compresser  store
  ventes/clients.json.k0                       compresser  store
  ventes/clients.json.k1                       compresser  lzma
  ventes/ventes.csv.c0         

### Lecture du résultat

Le nombre d'appels affiché est celui des appels **réellement partis** vers le modèle : le compteur est incrémenté juste avant l'envoi, dans `demander_decision`, et nulle part ailleurs. C'est une mesure, pas une estimation.

La recette se lit comme un arbre : les lignes `compresser` sont les feuilles — traitement final, irréversible — et les lignes `decouper` sont les nœuds internes. La couverture vérifiée à la fin de `trier` porte sur les racines, et elle est récursive : un nœud interne n'est couvert que si tous ses enfants le sont.

Les choix du modèle méritent d'être confrontés à ce que la structure annonçait. Sur un CSV dont la carte donne les six colonnes, un découpage `colonnes_csv` sépare une colonne de catégories d'une colonne de montants ; sur un binaire sans structure reconnue, la seule décision raisonnable est `compresser` — et `store` sur les octets aléatoires, puisque rien n'y est compressible. Le modèle ne le fait pas toujours, et le carnet ne corrige pas ses décisions : les chiffres de la comparaison sont les siens.

## Décompresser — sans aucun appel au modèle

C'est ici que la thèse du carnet se vérifie. La décompression **rejoue la recette à l'envers** : elle n'a besoin ni du modèle, ni de la carte du corpus, ni d'une heuristique. Chaque décision porte son inverse, et le recollage est déterministe.

Le compteur d'appels est remis à zéro **juste avant** la reconstruction, puis relu **juste après**. Un compteur à zéro n'est une preuve que si l'on sait qu'il aurait pu bouger : comme la même fonction de comptage est celle qui a produit le nombre d'appels du triage quelques cellules plus haut, un appel qui se glisserait ici serait vu.

La preuve finale n'est pas « ça décompresse » : c'est l'**égalité binaire**, fichier par fichier, avec les octets d'origine — comparés par empreinte pour être lisibles.

In [15]:
import hashlib


def reconstruire(sid: str, recette: dict, segments: dict) -> bytes:
    """Rejoue la recette a l'envers. Aucun appel au modele, aucune heuristique."""
    entree = recette.get(sid)
    if entree is None:
        raise CouvertureIncomplete(f"{sid} absent de la recette")
    if entree["op"] == "compresser":
        _, decompresser = CODECS[entree["codec"]]
        return decompresser(segments[sid].data)
    enfants = [reconstruire(e, recette, segments) for e in entree["enfants"]]
    return recoller(entree["mode"], enfants, entree["colle"])


COMPTEUR["appels_tri"] = COMPTEUR["appels"]  # le total du triage survit au reset
COMPTEUR["appels"] = 0          # remise a zero : le compteur devient l'oracle
reconstruits = {sid: reconstruire(sid, recette, resultat["segments"]) for sid in SEGMENTS}
appels_pendant_decompression = COMPTEUR["appels"]

egalites = []
for chemin in FICHIERS:
    relatif = chemin.relative_to(RACINE).as_posix()
    origine = chemin.read_bytes()
    obtenu = reconstruits[f"{relatif}#0"]
    egalites.append((relatif, origine == obtenu, hashlib.sha256(obtenu).hexdigest()[:16]))

for relatif, egal, empreinte in egalites:
    print(f"  {'IDENTIQUE' if egal else 'DIFFERENT':10s} {relatif:26s} sha256[:16]={empreinte}")

assert appels_pendant_decompression == 0, "la decompression a appele le modele"
assert all(egal for _, egal, _ in egalites), "au moins un fichier differe de son original"
print()
print(f"appels LLM pendant la decompression : {appels_pendant_decompression}")
print(f"{len(egalites)}/{len(egalites)} fichiers reconstruits a l'identique, "
      f"total {sum(len(v) for v in reconstruits.values())} o")

  IDENTIQUE  config/settings.json       sha256[:16]=b2b460fa50591305
  IDENTIQUE  data/mesures.bin           sha256[:16]=8ac1d0c4b56ecc39
  IDENTIQUE  data/mixte.dat             sha256[:16]=4c2587020e2dcfab
  IDENTIQUE  docs/notes.md              sha256[:16]=8be478aa358bbcc7
  IDENTIQUE  logs/app.log               sha256[:16]=9c82e5f6d750df2a
  IDENTIQUE  src/analyse.py             sha256[:16]=009b8b956d5e7700
  IDENTIQUE  ventes/clients.json        sha256[:16]=2dad6acb5cf0869d
  IDENTIQUE  ventes/ventes.csv          sha256[:16]=750e8428f0cbe10a

appels LLM pendant la decompression : 0
8/8 fichiers reconstruits a l'identique, total 110688 o


### Lecture du résultat

Tous les fichiers sont identiques à leurs originaux, **et** le compteur affiche zéro. Les deux ensemble sont la preuve : l'égalité binaire seule ne dirait pas si le modèle a été consulté, et le compteur seul ne dirait pas si la reconstruction est fidèle.

L'empreinte tronquée à seize caractères rend la comparaison lisible sans noyer la sortie ; la comparaison réelle, elle, porte sur les octets complets (`origine == obtenu`).

Notez ce que la recette ne contient **pas** : aucune donnée du corpus. Un décodeur qui posséderait le corpus d'origine pourrait toujours « reconstruire », et ne prouverait rien. Ici le décodeur n'a que les artefacts compressés, les identifiants et les colles de découpage.

## La thèse à réfuter : le triage bat-il le traitement indifférencié ?

Toute la construction qui précède ne vaut que si elle répond à une question mesurable. La voici, posée par le Concern à l'origine de ce carnet : **consacrer le modèle au triage et aux transformations de structure permet-il de mieux compresser qu'un traitement indifférencié ?**

Trois totaux sont mis en regard, tous en octets, tous incluant ce qu'un comparaison honnête doit inclure :

1. **triage** — somme des artefacts de toutes les feuilles, **plus** l'en-tête du dictionnaire, **plus** la recette sérialisée et compressée. Une recette gratuite n'existe pas : elle fait partie du fichier qu'il faut transmettre pour pouvoir décompresser.
2. **bloc indifférencié, même codec** — les fichiers mis bout à bout, compressés d'un seul tenant avec le codec que le triage a le plus employé. C'est le traitement naïf équitable.
3. **bloc indifférencié, meilleur codec choisi après coup** — on essaie les quatre codecs sur le tout et on garde le meilleur. C'est une **borne favorable au traitement naïf** : personne ne fait cela sans essayer tous les codecs sur tout le contenu, mais elle donne au camp d'en face tout le bénéfice du doute.

Le verdict est publié **même s'il est défavorable au triage**. Un carnet qui ne publierait que les cas où il gagne ne mesurerait rien.

In [16]:
import statistics


def taille_recette(recette: dict) -> int:
    """La recette doit voyager : son cout est compte, compresse comme elle le serait."""
    brut = json.dumps(recette, ensure_ascii=False, sort_keys=True).encode("utf-8")
    return len(zlib.compress(brut, 9))


def tout_en_un() -> bytes:
    return b"".join(p.read_bytes() for p in FICHIERS)


feuilles = [s for s in resultat["segments"].values()
            if recette[s.id]["op"] == "compresser"]
artefacts = sum(len(s.data) for s in feuilles)
cout_recette = taille_recette(recette)
cout_dico = len(EN_TETE_DICO)
total_triage = artefacts + cout_recette + cout_dico

# Codec majoritaire du triage : c'est le codec que le camp naif est en droit d'utiliser.
codecs_choisis = Counter(e["codec"] for e in recette.values() if e["op"] == "compresser")
codec_maj = codecs_choisis.most_common(1)[0][0]
bloc = tout_en_un()

total_naif_meme = len(CODECS[codec_maj][0](bloc))
totaux_par_codec = {nom: len(CODECS[nom][0](bloc)) for nom in CODECS}
meilleur = min(totaux_par_codec, key=totaux_par_codec.get)
total_naif_meilleur = totaux_par_codec[meilleur]

print(f"corpus complet                      : {len(bloc):8d} o")
print()
print(f"  triage LLM                        : {total_triage:8d} o "
      f"({artefacts} artefacts + {cout_recette} recette + {cout_dico} dictionnaire)")
print(f"  bloc indifferentie, {codec_maj:5s}        : {total_naif_meme:8d} o")
print(f"  bloc indifferentie, meilleur ({meilleur:4s}) : {total_naif_meilleur:8d} o")
print()
print(f"codecs choisis par le triage : {dict(codecs_choisis)}")
print(f"tailles par codec sur le bloc : {totaux_par_codec}")
print()
gain = total_naif_meme - total_triage
print(f"triage vs bloc/meme codec     : {gain:+8d} o  ({gain / total_naif_meme:+.1%})")
gain2 = total_naif_meilleur - total_triage
print(f"triage vs bloc/meilleur codec : {gain2:+8d} o  ({gain2 / total_naif_meilleur:+.1%})")
verdict = "LE TRIAGE GAGNE" if gain2 > 0 else "LE TRIAGE PERD"
print()
print(f"VERDICT : {verdict} face a la borne la plus favorable au traitement naif")

corpus complet                      :   110688 o

  triage LLM                        :    66940 o (66184 artefacts + 353 recette + 403 dictionnaire)
  bloc indifferentie, lzma         :    64888 o
  bloc indifferentie, meilleur (lzma) :    64888 o

codecs choisis par le triage : {'zlib': 3, 'store': 3, 'lzma': 6, 'dict': 4}
tailles par codec sur le bloc : {'zlib': 66341, 'bz2': 66724, 'lzma': 64888, 'store': 110688, 'dict': 66561}

triage vs bloc/meme codec     :    -2052 o  (-3.2%)
triage vs bloc/meilleur codec :    -2052 o  (-3.2%)

VERDICT : LE TRIAGE PERD face a la borne la plus favorable au traitement naif


### Lecture du résultat — le verdict, quel qu'il soit

Trois totaux, un verdict, et une conclusion qui n'est pas négociable avec le résultat.

**Ce qu'il faut regarder avant le verdict.** Le total du triage paie trois choses que le traitement naïf ne paie pas : la recette (une décision par segment, sérialisée et compressée), l'en-tête du dictionnaire, et le coût de découpage lui-même — une colonne de CSV extraite n'est plus contiguë, donc le recolle coûte de la recette. Ces coûts sont **dans** le total, pas à côté : c'est ce qui rend la comparaison honnête, et c'est aussi ce qui explique qu'un triage intelligent puisse perdre.

**Ce qu'un gain ou une perte signifie vraiment.** Un gain établi sur ce corpus dirait que la structure de ces fichiers — colonnes, clés, lignes de journal — est une redondance que les codecs génériques ne voient pas d'un bloc. Une perte dirait le contraire : que les trois codecs de la bibliothèque standard exploitent déjà ces régularités sur le bloc entier, et que le prix du triage n'est pas récupéré. Les deux verdicts sont des résultats ; seul un verdict fabriqué ne le serait pas.

### Exercice 3 — la borne qui manque : meilleur codec par fichier

La comparaison ci-dessus offre au traitement naïf **un seul codec pour tout le corpus**. Ce n'est pas la borne la plus généreuse qu'on puisse lui construire : un adversaire sérieux choisirait, pour chaque fichier, le codec qui le compresse le mieux — sans structure, sans modèle, mais avec un essai systématique.

**Objectif** : compléter `meilleur_par_fichier` pour qu'elle rende la taille totale obtenue en compressant **chaque fichier séparément** avec son meilleur codec, plus la taille de la table qui dit quel codec va avec quel fichier.

**Indice** — les fichiers sont dans `FICHIERS` ; leurs noms relatifs s'obtiennent par `chemin.relative_to(RACINE).as_posix()`. La table des choix se sérialise et se compresse exactement comme la recette, avec `taille_recette`.

**Étape 1** : pour chaque fichier, essayer tous les `CODECS` et retenir le plus petit. **Étape 2** : sommer ces minima. **Étape 3** : ajouter le coût de la table `{nom: codec}`.

**Question à trancher vous-même** : si le triage perd encore face à cette borne-là, qu'est-ce que cela dit de l'hypothèse du Concern — et qu'est-ce que cela ne dit pas ? (Un corpus de huit fichiers n'est pas un corpus de mille.)

In [17]:
def meilleur_par_fichier():
    """Exercice 3 : borne « meilleur codec par fichier », plus couteuse a etablir."""
    # TODO etudiant : suivre les trois etapes de l'enonce.
    return None


_borne = meilleur_par_fichier()
if _borne is None:
    print("Exercice 3 a completer : borne meilleur-codec-par-fichier")
else:
    print(f"bloc indifferentie, meilleur codec par fichier : {_borne} o")
    print(f"triage : {total_triage} o -> ecart {_borne - total_triage:+d} o")

Exercice 3 a completer : borne meilleur-codec-par-fichier


## L'entonnoir : le modèle fournit les infos de base, l'automatique fait le reste

Le triage précédent demande au modèle de choisir **chaque** opération. L'entonnoir renverse la répartition : le modèle ne fournit que ce que seul un modèle peut fournir — le format, le schéma, les repères qui délimitent les portions reconnues — et le pipeline exécute le gros œuvre de façon déterministe : il délimite, analyse, resérialise, canonise, choisit le meilleur codec **par portion**, et vérifie l'identité de l'artefact reconstruit.

Trois routes sont mesurées sur le même corpus, et la moins favorable au carnet est publiée avec les autres :

- **R0** — le brut : le meilleur codec sur le fichier entier, sans rien comprendre. C'est exactement la borne que demande l'exercice 3 ; l'entonnoir la calcule pour de bon, sur la chaîne complète et non pour un seul fichier ;
- **R1** — la canonisation : le document est réécrit sous sa forme **canonique** — un JSON sans indentation, par exemple — puis compressé. La route n'est comptée que si le document relu depuis cette forme est le **même** document : l'identité est sémantique, et elle est vérifiée, pas supposée ;
- **R2** — le schéma : découpage par sens (colonnes d'un CSV, clés d'un JSON, heures d'un journal, sections d'un markdown, définitions d'un source), meilleur codec par portion, **plus** le coût des métadonnées et de la recette.

Aucune des trois ne gagne partout, et c'est précisément pourquoi le carnet publie aussi l'**arbitrage** : le minimum des trois, fichier par fichier. Un pipeline réel ne choisit pas une stratégie, il mesure puis il choisit.

Un entonnoir qui perdrait partout ne démontrerait rien : on ne saurait pas si le mécanisme est mauvais ou si ces données-ci ne s'y prêtent pas. La cellule se termine donc par un **contrôle positif** — un CSV de deux mille lignes à colonnes faiblement cardinales, construit pour que le découpage par colonnes doive gagner. Les deux mesures sont publiées ensemble.

La provenance est le même geste que l'identité : un sérialiseur qui reproduit l'artefact à l'identique **est** un générateur candidat du document, et c'est la mesure d'identité qui l'autorise ou le rejette. Pour un média binaire, l'inférence du code générateur reste l'étape d'après — elle n'est pas démontrée ici, et le carnet le dit plutôt que de le laisser croire.

In [18]:
# L'ENTONNOIR : le modele ne fournit que les infos de base (format, schema, reperes) ;
# le gros oeuvre -- delimiter, analyser, reserialiser, canoniser, choisir le codec de
# chaque portion, verifier l'identite -- est automatique. Trois routes sont mesurees
# par fichier, et l'arbitrage choisit entre elles.

def meilleur_codec(data: bytes) -> tuple[str, int]:
    """Le meilleur codec pour CES octets, parmi ceux que la recette connait."""
    tailles = {nom: len(CODECS[nom][0](data)) for nom in CODECS}
    nom = min(tailles, key=tailles.get)
    return nom, tailles[nom]


def forme_canonique(modele: ModeleDocument) -> bytes:
    """Meme document, ecriture normalisee -- c'est la que le serialiseur gagne sa place.

    L'indentation d'un JSON est de la redondance qu'un codec devine mal et qu'un
    serialiseur supprime exactement. Le document reste le meme ; seule sa forme change.
    """
    if modele.format == "json":
        return json.dumps(modele.contenu, separators=(",", ":"),
                          ensure_ascii=False).encode("utf-8")
    if modele.format == "csv":
        tampon = io.StringIO()
        graveur = csv.writer(tampon, lineterminator="\n")
        graveur.writerow(modele.contenu["entetes"])
        graveur.writerows(modele.contenu["lignes"])
        return tampon.getvalue().encode("utf-8")
    if modele.format == "journal":
        return serialiser_journal(modele)
    if modele.format == "markdown":
        return modele.contenu["texte"].encode("utf-8")
    return ast.unparse(modele.contenu).encode("utf-8")


def identite_semantique(modele: ModeleDocument, canonique: bytes) -> dict:
    """La forme canonique est-elle le MEME document ? On le verifie, on ne le suppose pas."""
    analyseur = ANALYSEURS.get(modele.format)
    if analyseur is None:
        return {"identique": False, "methode": "aucun analyseur"}
    relu = analyseur(canonique)
    if modele.format == "python":
        meme = (ast.dump(relu.contenu, include_attributes=False)
                == ast.dump(modele.contenu, include_attributes=False))
        methode = "arbres AST egaux"
    else:
        meme = relu.contenu == modele.contenu
        methode = f"modele {modele.format} relu egal"
    return {"identique": bool(meme), "methode": methode}


def _comprimer_feuille(seg: Segment, recette: dict, finals: dict,
                       meta: MetadonneesSegment) -> None:
    """Une portion recoit son traitement final : le meilleur codec pour ELLE."""
    nom, _ = meilleur_codec(seg.data)
    compresse, _ = CODECS[nom]
    seg.data = compresse(seg.data)
    seg.genre = f"{seg.genre}/compresse:{nom}"
    finals[seg.id] = seg
    recette[seg.id] = {"op": "compresser", "codec": nom, "metadonnees": meta.vers_dict()}


def entonnoir_fichier(seg: Segment) -> dict:
    """Un fichier : reconnu, decrit, canonise, delimite, compresse -- et sa recette chiffree."""
    r0_nom, r0 = meilleur_codec(seg.data)          # mesure AVANT toute mutation
    reconnu = reconnaitre_format(seg.data)
    analyseur = ANALYSEURS.get(reconnu["format"])
    modele = analyseur(seg.data) if analyseur else None
    # Le schema et la provenance vivent sur la RACINE : les recopier sur chaque feuille
    # ferait payer la description autant de fois qu'il y a de portions.
    meta_racine = MetadonneesSegment(format=reconnu["format"], schema=schema_document(modele),
                                     role="racine", encodage="utf-8" if modele else None)
    r1 = None
    if modele is not None:
        canonique = forme_canonique(modele)
        verdict_sem = identite_semantique(modele, canonique)
        verdict_bin = identite_artefact(lambda: SERIALISEURS[modele.format](modele), seg.data)
        # La provenance ne se declare pas, elle se mesure : un serialiseur qui redonne
        # les octets est un generateur candidat BINAIRE ; un serialiseur qui ne redonne
        # que le meme document est un generateur SEMANTIQUE -- et le carnet dit lequel.
        meta_racine.provenance = {"forme": "serialiseur",
                                  "binaire": verdict_bin["identique"],
                                  "semantique": verdict_sem["identique"],
                                  "sha_artefact": verdict_bin["sha_artefact"]}
        # R1 n'est comptee que si la canonisation preserve le document.
        if verdict_sem["identique"]:
            r1 = meilleur_codec(canonique)[1]
    recette, finals = {}, {}
    base = seg.id.rsplit("#", 1)[0]
    if reconnu["format"] == "csv":
        enfants, colle = decouper(seg, "colonnes_csv", base)
        metas = [MetadonneesSegment(format="csv", role=e.genre.split(":", 1)[-1],
                                    schema={"champ": e.genre.split(":", 1)[-1]})
                 for e in enfants]
    elif reconnu["format"] == "json":
        # La RACINE decide de l'operation, comme partout ailleurs : un tableau
        # n'a pas de cles. Le mode etait ecrit en dur ici.
        racine_json = json.loads(seg.data.decode("utf-8"))
        mode_json = "cles_json" if isinstance(racine_json, dict) else "elements_json"
        enfants, colle = decouper(seg, mode_json, base)
        metas = [MetadonneesSegment(format="json", role=e.genre.split(":", 1)[-1],
                                    schema={"champ": e.genre.split(":", 1)[-1]})
                 for e in enfants]
    else:
        reperes = reperes_depuis_schema(seg, reconnu, modele)
        if reperes:
            enfants, metas, colle = segments_par_reperes(seg, reperes, base)
            for meta in metas:
                meta.format = reconnu["format"]
        else:
            enfants, metas, colle = [], [], None
    if enfants:
        recette[seg.id] = {"op": "decouper", "mode": colle["mode"], "colle": colle,
                           "enfants": [e.id for e in enfants],
                           "metadonnees": meta_racine.vers_dict()}
        for enfant, meta in zip(enfants, metas):
            _comprimer_feuille(enfant, recette, finals, meta)
    else:
        _comprimer_feuille(seg, recette, finals, meta_racine)
    artefacts = sum(len(e.data) for e in finals.values())
    return {"reconnu": reconnu, "schema": schema_document(modele), "recette": recette,
            "finals": finals, "R0": r0, "R1": r1,
            "R2": artefacts + taille_recette(recette), "codec_brut": r0_nom,
            "portions": len(finals), "artefacts": artefacts,
            "recette_octets": taille_recette(recette),
            "arbitrage": min(v for v in (r0, r1, artefacts + taille_recette(recette))
                             if v is not None)}


CHEMINS = {f"{chemin.relative_to(RACINE).as_posix()}#0": chemin for chemin in FICHIERS}
SEGMENTS_FRAIS = {sid: Segment(sid, chemin.read_bytes(), chemin.suffix.lower())
                  for sid, chemin in CHEMINS.items()}
RESULTATS_ENTONNOIR = {sid: entonnoir_fichier(seg) for sid, seg in SEGMENTS_FRAIS.items()}

RECETTE_ENTONNOIR, FINALS_ENTONNOIR = {}, {}
for res in RESULTATS_ENTONNOIR.values():
    RECETTE_ENTONNOIR.update(res["recette"])
    FINALS_ENTONNOIR.update(res["finals"])

# Identite de l'entonnoir : la meme preuve que le triage, rejouee sans modele.
COMPTEUR["appels_avant_entonnoir"] = COMPTEUR["appels"]
RECONSTRUITS_ENTONNOIR = {sid: reconstruire(sid, RECETTE_ENTONNOIR, FINALS_ENTONNOIR)
                          for sid in SEGMENTS_FRAIS}
IDENTITES_ENTONNOIR = {sid: RECONSTRUITS_ENTONNOIR[sid] == CHEMINS[sid].read_bytes()
                       for sid in SEGMENTS_FRAIS}
assert COMPTEUR["appels"] == COMPTEUR["appels_avant_entonnoir"], "l'entonnoir a appele le modele"
assert all(IDENTITES_ENTONNOIR.values()), f"reconstruction non identique : {IDENTITES_ENTONNOIR}"

print(f"{'fichier':26s} {'format':9s} {'R0 brut':>8s} {'R1 canon':>9s} {'R2 schema':>10s} "
      f"{'arbitre':>8s} {'port.':>6s}")
totaux = {"R0": 0, "R1": 0, "R2": 0, "arbitre": 0, "artefacts": 0,
          "R1_fichiers": 0, "R0_r1": 0}
for sid, res in RESULTATS_ENTONNOIR.items():
    relatif = sid.rsplit("#", 1)[0]
    totaux["R0"] += res["R0"]
    totaux["R2"] += res["R2"]
    totaux["arbitre"] += res["arbitrage"]
    totaux["artefacts"] += res["artefacts"]
    if res["R1"] is not None:
        totaux["R1"] += res["R1"]
        totaux["R0_r1"] += res["R0"]
        totaux["R1_fichiers"] += 1
    r1_affiche = f"{res['R1']:9d}" if res["R1"] is not None else f"{'--':>9s}"
    print(f"  {relatif:26s} {res['reconnu']['format']:9s} {res['R0']:8d} {r1_affiche} "
          f"{res['R2']:10d} {res['arbitrage']:8d} {res['portions']:6d}")

print()
print(f"  total R0 (brut, meilleur codec)            : {totaux['R0']:8d} o")
print(f"  total R1 (canonique, {totaux['R1_fichiers']} fichier(s))    : {totaux['R1']:8d} o")
print(f"  total R2 (schema + recette + metadonnees)  : {totaux['R2']:8d} o "
      f"({totaux['artefacts']} artefacts + {totaux['R2'] - totaux['artefacts']} recette)")
print(f"  total arbitre (min des trois, par fichier) : {totaux['arbitre']:8d} o")
print()
# R1 ne porte que sur les fichiers canonisables : on le compare au R0 de CES
# fichiers-la, jamais au total des huit -- sinon la comparaison est fausse.
print(f"  canonisation vs brut  : {totaux['R0_r1'] - totaux['R1']:+8d} o "
      f"({(totaux['R0_r1'] - totaux['R1']) / totaux['R0_r1']:+.1%} "
      f"sur {totaux['R1_fichiers']} fichier(s), {totaux['R0_r1']} o)")
print(f"  entonnoir vs brut     : {totaux['R0'] - totaux['R2']:+8d} o "
      f"({(totaux['R0'] - totaux['R2']) / totaux['R0']:+.1%})")
print(f"  arbitrage vs brut     : {totaux['R0'] - totaux['arbitre']:+8d} o "
      f"({(totaux['R0'] - totaux['arbitre']) / totaux['R0']:+.1%})")
print(f"  identite : {sum(IDENTITES_ENTONNOIR.values())}/{len(IDENTITES_ENTONNOIR)} fichiers "
      f"reconstruits a l'identique, "
      f"{COMPTEUR['appels'] - COMPTEUR['appels_avant_entonnoir']} appel(s) au modele")

# CONTROLE POSITIF -- sans lui, un entonnoir qui perd partout ne prouverait rien : on ne
# saurait pas si le mecanisme est mauvais ou si ces donnees-ci ne s'y pretent pas. Ce
# temoin est un CSV de 2 000 lignes a colonnes faiblement cardinales : le decoupage par
# colonnes DOIT y gagner, et de beaucoup.
_regions = ["nord", "sud", "est", "ouest", "centre"]
_produits = [f"p{i:03d}" for i in range(200)]
_canaux = ["web", "magasin", "partenaire"]
_tampon = io.StringIO()
_graveur = csv.writer(_tampon, lineterminator="\n")
_graveur.writerow(["date", "region", "produit", "quantite", "prix", "canal"])
for _i in range(2000):
    _graveur.writerow([f"2026-{(_i % 12) + 1:02d}-{(_i % 28) + 1:02d}", _regions[_i % 5],
                       _produits[(_i * 7) % 200], str(1 + _i % 9),
                       f"{(10 + (_i % 400) / 100):.2f}", _canaux[_i % 3]])
_OCTETS_TEMOIN = _tampon.getvalue().encode("utf-8")
_TEMOIN = Segment("temoin/colonnes.csv#0", _OCTETS_TEMOIN, "csv")
_RES_TEMOIN = entonnoir_fichier(_TEMOIN)
_RECONSTRUIT_TEMOIN = reconstruire(_TEMOIN.id, _RES_TEMOIN["recette"], _RES_TEMOIN["finals"])
assert _RECONSTRUIT_TEMOIN == _OCTETS_TEMOIN, "le temoin ne se reconstruit pas a l'identique"
print()
print(f"  controle positif (CSV de 2 000 lignes, {len(_OCTETS_TEMOIN)} o) :")
print(f"    R0 brut {_RES_TEMOIN['R0']:7d} o -> R2 colonnes {_RES_TEMOIN['R2']:7d} o  "
      f"({_RES_TEMOIN['R0'] - _RES_TEMOIN['R2']:+d} o, "
      f"{(_RES_TEMOIN['R0'] - _RES_TEMOIN['R2']) / _RES_TEMOIN['R0']:+.1%}) sur "
      f"{_RES_TEMOIN['portions']} portions, identite verifiee")
print("    le mecanisme sait gagner : c'est la FORME des donnees qui decide, pas le mecanisme")

_exemple = RECETTE_ENTONNOIR["config/settings.json#0"]
print()
print(f"  metadonnees de config/settings.json : "
      f"{json.dumps(_exemple['metadonnees'], ensure_ascii=False)[:240]}")
print(f"  provenance verifiee par l'identite  : "
      f"{json.dumps(_exemple['metadonnees']['provenance'], ensure_ascii=False)}")

fichier                    format     R0 brut  R1 canon  R2 schema  arbitre  port.
  config/settings.json       json            84        71        445       71      3
  data/mesures.bin           binaire      40000        --      40130    40000      1
  data/mixte.dat             binaire      20256        --      20385    20256      1
  docs/notes.md              markdown       300       300       1638      300      9
  logs/app.log               journal       1518      1518       2904     1518      4
  src/analyse.py             python         544       544       1296      544      5
  ventes/clients.json        json           536       498        919      498      2
  ventes/ventes.csv          csv            924       924       1112      924      6

  total R0 (brut, meilleur codec)            :    64162 o
  total R1 (canonique, 6 fichier(s))    :     3855 o
  total R2 (schema + recette + metadonnees)  :    68829 o (65797 artefacts + 3032 recette)
  total arbitre (min des trois, pa


  controle positif (CSV de 2 000 lignes, 73770 o) :
    R0 brut    6838 o -> R2 colonnes    1746 o  (+5092 o, +74.5%) sur 6 portions, identite verifiee
    le mecanisme sait gagner : c'est la FORME des donnees qui decide, pas le mecanisme

  metadonnees de config/settings.json : {"format": "json", "schema": {"format": "json", "structure": {"type": "objet", "cles": {"seuil": {"type": "reel"}, "reessais": {"type": "entier"}, "regions": {"type": "liste", "longueur": 5, "element": {"type": "str"}}}}}, "role": "racine",
  provenance verifiee par l'identite  : {"forme": "serialiseur", "binaire": true, "semantique": true, "sha_artefact": "b2b460fa50591305"}


### La même chaîne, mais pilotée par le modèle

Reste le point que l'entonnoir n'a pas encore montré : les infos de base, c'est le **modèle** qui les fournit, par *function calling*. Le plugin ci-dessous n'expose que trois fonctions — reconnaître le format, décrire le schéma, désigner les repères — et aucune d'elles ne compresse ni ne découpe. Le modèle ne peut donc pas faire le gros œuvre même s'il le voulait : la séparation est dans l'interface, pas dans la consigne.

Le repère proposé au modèle est **dérivé du document** — la date du premier enregistrement conforme du journal, lue sur le flux — et non écrit dans le code du carnet. Si le modèle rend un repère qui ne désigne rien, la fonction le refuse en le nommant et le modèle est relancé une fois ; passé ce point, la cellule échoue plutôt que de fabriquer un résultat.

In [19]:
class PluginBaseInfo:
    """Les trois infos de base que le MODELE fournit. Aucun gros oeuvre ici."""

    def __init__(self, etat: dict):
        self.etat = etat

    @kernel_function(name="reconnaitre_format",
                     description=("Rend le format du segment, mesure sur ses octets, "
                                  "avec sa methode et sa confiance."))
    def reconnaitre_format(self) -> str:
        etat = self.etat
        etat["appels"].append("reconnaitre_format")
        return json.dumps(etat["reconnu"], ensure_ascii=False)

    @kernel_function(name="decrire_schema",
                     description=("Rend le schema du modele du document : colonnes, cles, "
                                  "champs, titres, symboles."))
    def decrire_schema(self) -> str:
        etat = self.etat
        etat["appels"].append("decrire_schema")
        return json.dumps(etat["schema"], ensure_ascii=False)[:1500]

    @kernel_function(
        name="designer_reperes",
        description=("Designe un repere qui delimite les portions reconnues du segment. "
                     "Recopie EXACTEMENT un couple (type_repere, valeur) de "
                     "`reperes_proposes`, et donne une etiquette courte a la portion "
                     "qui s'ouvre."),
    )
    def designer_reperes(self, type_repere: str, valeur: str, etiquette: str) -> str:
        etat = self.etat
        etat["appels"].append(f"designer_reperes({type_repere},{valeur!r},{etiquette})")
        try:
            repere = Repere(type_repere, None if type_repere == "titre_markdown" else valeur,
                            etiquette)
            positions = repere.positions(etat["segment"].data)
        except ValueError as erreur:
            return f"refuse : {erreur}"
        etat["repere"] = repere
        return f"accepte : {len(positions)} frontiere(s) pour {etiquette!r}"


async def demander_infos_base(seg: Segment, reconnu: dict, schema: dict) -> dict:
    """Un aller-retour : le modele donne les infos de base, le pipeline fait le reste."""
    etat = {"segment": seg, "reconnu": reconnu, "schema": schema, "repere": None, "appels": []}
    kernel = Kernel()
    kernel.add_service(OpenAIChatCompletion(ai_model_id=MODELE, async_client=aoai,
                                            service_id=SERVICE))
    kernel.add_plugin(PluginBaseInfo(etat), plugin_name="base")
    propositions = [r.vers_dict() for r in reperes_depuis_schema(seg, reconnu)]
    reglages = OpenAIPromptExecutionSettings(
        max_tokens=3000, temperature=0.0,
        function_choice_behavior=FunctionChoiceBehavior.Auto(
            auto_invoke=True, maximum_auto_invoke_attempts=4),
    )
    service = kernel.get_service(SERVICE)
    historique = ChatHistory()
    historique.add_system_message(
        "Tu fournis les INFORMATIONS DE BASE d'un segment, rien de plus. Tu appelles les "
        "fonctions du plugin `base` : reconnaitre_format, decrire_schema, puis "
        "designer_reperes. Pour designer_reperes, tu recopies EXACTEMENT un couple "
        "(type_repere, valeur) de la liste `reperes_proposes`. Tu ne compresses rien et tu "
        "ne decoupes rien : le pipeline s'en charge."
    )
    historique.add_user_message(
        f"Segment : {json.dumps(vue(seg), ensure_ascii=False)}\n"
        f"reperes_proposes : {json.dumps(propositions, ensure_ascii=False)}\n"
        "Appelle les trois fonctions, en terminant par designer_reperes."
    )
    COMPTEUR["appels"] += 1
    await service.get_chat_message_contents(chat_history=historique, settings=reglages,
                                            kernel=kernel)
    if etat["repere"] is None:
        # Un seul relancement, avec le refus sous les yeux : c'est la meme discipline que
        # le triage. Passe ce point, la cellule echoue -- un repli silencieux fabriquerait
        # un resultat que rien n'a produit.
        historique.add_user_message(
            "Aucun repere valide n'a ete designe. Appelle designer_reperes en recopiant "
            f"exactement un couple de : {json.dumps(propositions, ensure_ascii=False)}")
        COMPTEUR["appels"] += 1
        await service.get_chat_message_contents(chat_history=historique, settings=reglages,
                                                kernel=kernel)
    if etat["repere"] is None:
        raise RuntimeError(
            f"le modele n'a designe aucun repere valide pour {seg.id} -- passe rouge ; "
            f"appels observes : {etat['appels']}")
    return etat


SID_LOG = "logs/app.log#0"
# Segment neuf, relu du disque : l'entonnoir a pu compresser le sien en place.
_SEG_LOG = Segment(SID_LOG, CHEMINS[SID_LOG].read_bytes(), ".log")
_infos = await demander_infos_base(_SEG_LOG,
                                   RESULTATS_ENTONNOIR[SID_LOG]["reconnu"],
                                   RESULTATS_ENTONNOIR[SID_LOG]["schema"])
_enfants, _metas, _colle = segments_par_reperes(_SEG_LOG, [_infos["repere"]],
                                                "logs/app.log/modele")
_recette_modele = {SID_LOG: {"op": "decouper", "mode": "reperes", "colle": _colle,
                             "enfants": [e.id for e in _enfants],
                             "metadonnees": {"format": "journal", "role": "racine"}}}
_finals_modele: dict = {}
for _enfant, _meta in zip(_enfants, _metas):
    _meta.format = "journal"
    _comprimer_feuille(_enfant, _recette_modele, _finals_modele, _meta)
_obtenu = reconstruire(SID_LOG, _recette_modele, _finals_modele)
_original = CHEMINS[SID_LOG].read_bytes()
_taille_modele = sum(len(e.data) for e in _finals_modele.values())

print(f"  modele servi       : {MODELE}")
print(f"  fonctions appelees : {_infos['appels']}")
print(f"  format (mesure)    : {_infos['reconnu']['format']} ({_infos['reconnu']['methode']})")
print(f"  repere designe     : {_infos['repere']}")
print(f"  portions produites : {len(_enfants)}")
print(f"  identite           : {'IDENTIQUE' if _obtenu == _original else 'DIFFERENTE'} "
      f"({len(_obtenu)} o reconstruits sur {len(_original)})")
print(f"  taille             : {_taille_modele} o compresses contre "
      f"{RESULTATS_ENTONNOIR[SID_LOG]['R0']} o au meilleur codec du fichier entier")
assert _obtenu == _original, "la segmentation designee par le modele ne se reconstruit pas"
print(f"  appels LLM         : "
      f"{COMPTEUR['appels'] - COMPTEUR['appels_avant_entonnoir']} "
      f"(les infos de base ; la reconstruction n'en fait aucun)")

  modele servi       : qwen3.6-35b-a3b
  fonctions appelees : ['reconnaitre_format', 'decrire_schema', "designer_reperes(ligne_regex,'^2026-11-15T\\\\d{2}:00:00Z',heure)"]
  format (mesure)    : journal (horodatage ISO en debut de ligne)
  repere designe     : <repere ligne_regex 'heure'>
  portions produites : 4
  identite           : IDENTIQUE (14194 o reconstruits sur 14194)
  taille             : 2436 o compresses contre 1518 o au meilleur codec du fichier entier
  appels LLM         : 1 (les infos de base ; la reconstruction n'en fait aucun)


### Lecture du résultat

Trois choses sortent de ce tableau, et la première n'est pas celle qu'on attendait.

La **canonisation** fait ce qu'elle promet, mais seulement là où il y a de la forme à jeter : un JSON indenté perd son indentation, qu'un codec devine mal, et la route gagne ; sur les formats dont l'écriture est déjà minimale, elle ne change rien et le dit.

Le **découpage par portions perd** sur ce corpus. Ce n'est pas un défaut du mécanisme, c'est une propriété de ces données : les fichiers font quelques kilooctets, ils sont homogènes, et découper un texte homogène en neuf portions remplace une fenêtre de compression partagée par neuf fenêtres séparées, chacune payant son en-tête et sa recette. Le carnet publie la perte au lieu de la maquiller — et c'est l'arbitrage qui rattrape : fichier par fichier, il garde la meilleure des trois routes, donc il ne perd jamais.

Le **contrôle positif** montre l'autre face : sur un CSV de deux mille lignes à colonnes faiblement cardinales, le même code, sans une ligne de change, gagne très largement. Ce n'est donc pas « le découpage ne marche pas », c'est « le découpage ne marche pas **sur ces huit fichiers-là** » — et c'est la forme des données qui décide, pas le mécanisme.

L'identité, elle, ne bouge pas : tous les fichiers se reconstruisent à l'identique, et zéro appel au modèle a été consommé pour y arriver. Le dernier bloc est la partie **function calling** : le modèle a reçu la carte du segment — format reconnu, schéma, repères proposés — et a désigné un repère par un appel d'outil ; le pipeline a ensuite découpé, compressé, reconstruit et vérifié sans lui. Une segmentation désignée par le modèle qui ne se reconstruirait pas ferait échouer la cellule.

## Inférer le générateur d'un média, et le proposer

La conclusion de ce carnet écrit ce qui n'était **pas** démontré : inférer le code qui a produit un média, le proposer, et vérifier qu'il reproduit l'artefact. Le report était fondé — sur une image sortie d'un modèle de diffusion, aucun générateur n'est rejouable, donc `identite_artefact` n'aurait rien à comparer.

Il existe en revanche une famille de médias où la question est **décidable** : ceux dont les octets sont une fonction pure de quelques nombres. Une onde synthétisée en est le cas d'école — quatre paramètres suffisent à reproduire le fichier au bit près.

La section suit l'entonnoir du carnet, dans le même ordre qu'ailleurs : le pipeline mesure, le modèle propose, le pipeline vérifie.

1. le pipeline **génère** un média déterministe et l'écrit sur disque ;
2. `reconnaitre_format` lit son conteneur **sur les octets** (RIFF/WAVE) — renommé en `.bin`, il reste un WAV, comme le CSV renommé plus haut ;
3. le pipeline **mesure** les infos de base : taux, nombre d'échantillons, durée, amplitude crête, période en échantillons, premiers échantillons ;
4. le modèle **propose** les paramètres du générateur par appel de fonction ;
5. le pipeline **synthétise** avec ces paramètres et compare les octets à l'artefact.

Ce qui reste hors de portée est dit à la fin de la section, et mesuré : le garde d'identité prouve qu'un code **reproduit** l'artefact, il ne prouve pas qu'il l'a **produit**.

In [20]:
import struct
import wave


# Un media DETERMINISTE : ses octets sont une fonction pure de quatre nombres. C'est la
# condition qui rend la provenance decidable -- sur une image sortie d'un modele de
# diffusion, le generateur n'est pas rejouable et le garde d'identite serait vide.
def synthese_wav(frequence_hz: float, amplitude_crete: float,
                 taux_echantillonnage: int, duree_s: float) -> bytes:
    """Une sinusoide pure, phase nulle, PCM 16 bits signe mono, conteneur WAV."""
    n = int(round(taux_echantillonnage * duree_s))
    echantillons = [int(round(amplitude_crete * 32767
                              * math.sin(2 * math.pi * frequence_hz * i / taux_echantillonnage)))
                    for i in range(n)]
    tampon = io.BytesIO()
    with wave.open(tampon, "wb") as graveur:
        graveur.setnchannels(1)
        graveur.setsampwidth(2)
        graveur.setframerate(taux_echantillonnage)
        graveur.writeframes(struct.pack(f"<{n}h", *echantillons))
    return tampon.getvalue()


def _periode_echantillons(echantillons: list[int]) -> int | None:
    """Le plus petit decalage non nul ou la suite se reproduit EXACTEMENT sur la fenetre.

    Mesure, pas devinette : une valeur n'est rendue que si elle tient sur les n-lag
    comparaisons ; `None` dit qu'aucune periode entiere n'existe, au lieu de rendre une
    valeur approchee qu'on prendrait pour la reponse.
    """
    n = len(echantillons)
    for lag in range(1, n // 2 + 1):
        if all(echantillons[i] == echantillons[i + lag] for i in range(n - lag)):
            return lag
    return None


def faits_media(data: bytes) -> dict:
    """Les infos de base d'un WAV, MESUREES sur les octets -- rien n'est repris d'un nom."""
    with wave.open(io.BytesIO(data)) as lecteur:
        taux = lecteur.getframerate()
        canaux = lecteur.getnchannels()
        largeur = lecteur.getsampwidth()
        n = lecteur.getnframes()
        brut = lecteur.readframes(n)
    echantillons = list(struct.unpack(f"<{n}h", brut))
    periode = _periode_echantillons(echantillons)
    valeur_max = max(abs(v) for v in echantillons)
    return {
        "format_reconnu": reconnaitre_format(data),
        "conteneur": "RIFF/WAVE",
        "codec": "pcm entier 16 bits signe, petit-boutiste",
        "canaux": canaux,
        "largeur_octets": largeur,
        "taux_echantillonnage_hz": taux,
        "n_echantillons": n,
        "duree_s": round(n / taux, 6),
        "valeur_max_absolue": valeur_max,
        "amplitude_crete": round(valeur_max / 32767, 3),
        "periode_echantillons": periode,
        "premiers_echantillons": echantillons[:16],
    }


# Le corpus de la passe ne bouge pas : ces deux medias vivent dans un sous-repertoire
# cree ici, apres la moisson -- aucun total du carnet ne les compte.
MEDIA = RACINE / "media"
MEDIA.mkdir(exist_ok=True)
_ARTEFACT_A = synthese_wav(500.0, 0.5, 8000, 0.25)
_ARTEFACT_B = synthese_wav(250.0, 0.5, 8000, 0.25)
(MEDIA / "onde_a.wav").write_bytes(_ARTEFACT_A)
(MEDIA / "onde_b.wav").write_bytes(_ARTEFACT_B)
_FAITS_A = faits_media((MEDIA / "onde_a.wav").read_bytes())

# Le nom ne fait pas le format : le meme fichier, renomme, reste un WAV.
_MEDIA_RENOMME = MEDIA / "onde_a_sans_extension.bin"
_MEDIA_RENOMME.write_bytes(_ARTEFACT_A)
_RECONNU_RENOMME = reconnaitre_format(_MEDIA_RENOMME.read_bytes())
assert _RECONNU_RENOMME["format"] == "wav", _RECONNU_RENOMME

# La synthese est une fonction : deux appels identiques rendent les memes octets.
assert synthese_wav(500.0, 0.5, 8000, 0.25) == _ARTEFACT_A
assert _ARTEFACT_A != _ARTEFACT_B

print(f"media A : {len(_ARTEFACT_A)} o, format reconnu {_RECONNU_RENOMME['format']} "
      f"({_RECONNU_RENOMME['methode']}, preuve {_RECONNU_RENOMME['preuve'][:16]}...)")
print(f"  renomme en .bin : toujours {_RECONNU_RENOMME['format']}")
print(f"  faits mesures   : {json.dumps({k: v for k, v in _FAITS_A.items() if k != 'premiers_echantillons'}, ensure_ascii=False)}")
print(f"  periode mesuree : {_FAITS_A['periode_echantillons']} echantillons -> "
      f"frequence deduite {_FAITS_A['taux_echantillonnage_hz'] / _FAITS_A['periode_echantillons']:.1f} Hz")

media A : 4044 o, format reconnu wav (signature riff, preuve 52494646c40f0000...)
  renomme en .bin : toujours wav
  faits mesures   : {"format_reconnu": {"format": "wav", "confiance": 1.0, "methode": "signature riff", "preuve": "52494646c40f000057415645"}, "conteneur": "RIFF/WAVE", "codec": "pcm entier 16 bits signe, petit-boutiste", "canaux": 1, "largeur_octets": 2, "taux_echantillonnage_hz": 8000, "n_echantillons": 2000, "duree_s": 0.25, "valeur_max_absolue": 16384, "amplitude_crete": 0.5, "periode_echantillons": 16}
  periode mesuree : 16 echantillons -> frequence deduite 500.0 Hz


### Le modèle propose, le pipeline synthétise et compare

Le plugin ne rend pas les octets : il rend un **verdict**, écart situé compris, pour que le modèle puisse se corriger dans le même tour. Une proposition n'est acceptée que si les octets rejoués sont identiques à l'artefact — il n'y a pas d'à-peu-près, pas de seuil de tolérance, pas de similarité spectrale.

Deux relances sont permises, pas davantage : passé ce point la cellule échoue. Un repli silencieux fabriquerait un résultat que rien n'a produit — c'est la même discipline que le triage.

Les trois contrôles qui suivent disent ce que le garde vaut : le générateur retenu ne reproduit **pas** un autre média, un paramètre voisin ne passe **pas**, et un générateur qui rend l'artefact tel quel **passe** — l'identité est nécessaire, pas suffisante.

In [21]:
class PluginGenerateurMedia:
    """Le modele propose les parametres du generateur ; le pipeline synthetise et compare.

    Le plugin ne rend jamais l'artefact : il rend le verdict d'identite, ecart situe
    compris, pour que le modele se corrige dans le meme tour. Une proposition acceptee
    est celle dont les octets rejoues sont identiques -- aucun seuil de tolerance.
    """

    def __init__(self, etat: dict):
        self.etat = etat

    @kernel_function(
        name="mesurer_media",
        description=("Rend les faits mesures sur les octets du media : conteneur, codec, "
                     "taux, nombre d'echantillons, duree, amplitude crete, periode en "
                     "echantillons et premiers echantillons."),
    )
    def mesurer_media(self) -> str:
        self.etat["appels"].append("mesurer_media")
        return json.dumps(self.etat["faits"], ensure_ascii=False)

    @kernel_function(
        name="proposer_generateur",
        description=("Propose les parametres d'une synthese sinusoidale : frequence en Hz, "
                     "amplitude crete dans [0, 1], taux d'echantillonnage en Hz, duree en "
                     "secondes. Le pipeline synthetise avec ces quatre nombres et compare "
                     "les octets produits a l'artefact."),
    )
    def proposer_generateur(self, frequence_hz: float, amplitude_crete: float,
                            taux_echantillonnage: float, duree_s: float) -> str:
        parametres = {"frequence_hz": float(frequence_hz),
                      "amplitude_crete": float(amplitude_crete),
                      "taux_echantillonnage": int(round(taux_echantillonnage)),
                      "duree_s": float(duree_s)}
        verdict = identite_artefact(lambda: synthese_wav(**parametres), self.etat["artefact"])
        self.etat["propositions"].append({"parametres": parametres, "verdict": verdict})
        if verdict["identique"]:
            self.etat["retenu"] = {"parametres": parametres, "verdict": verdict}
            return "accepte : les octets rejoues sont identiques a l'artefact"
        return (f"refuse : les octets different -- {verdict['ecart']}. "
                "Reprends les faits mesures et repropose : la frequence se deduit de "
                "`periode_echantillons` et de `taux_echantillonnage_hz`.")


async def demander_generateur(artefact: bytes, faits: dict, essais: int = 2) -> dict:
    """L'entonnoir du media : le modele propose, le pipeline verifie, l'ecart repart."""
    etat = {"artefact": artefact, "faits": faits, "propositions": [], "retenu": None,
            "appels": []}
    kernel = Kernel()
    kernel.add_service(OpenAIChatCompletion(ai_model_id=MODELE, async_client=aoai,
                                            service_id=SERVICE))
    kernel.add_plugin(PluginGenerateurMedia(etat), plugin_name="media")
    reglages = OpenAIPromptExecutionSettings(
        max_tokens=3000, temperature=0.0,
        function_choice_behavior=FunctionChoiceBehavior.Auto(
            auto_invoke=True, maximum_auto_invoke_attempts=6),
    )
    service = kernel.get_service(SERVICE)
    historique = ChatHistory()
    historique.add_system_message(
        "Tu inferes le GENERATEUR d'un media a partir de mesures prises sur ses octets. "
        "Tu appelles les fonctions du plugin `media` : `mesurer_media` pour lire les faits, "
        "puis `proposer_generateur` avec les quatre parametres d'une synthese sinusoidale "
        "(frequence en Hz, amplitude crete, taux d'echantillonnage en Hz, duree en secondes). "
        "Tu ne reponds pas en texte libre et tu ne devines pas : la frequence se deduit de "
        "`periode_echantillons` et de `taux_echantillonnage_hz`, l'amplitude de "
        "`amplitude_crete`, la duree de `duree_s`. Si une proposition est refusee, lis "
        "l'ecart rendu et repropose."
    )
    historique.add_user_message(
        "Un media a ete produit par un generateur inconnu de la famille : sinusoide pure, "
        "phase nulle, PCM 16 bits signe mono, conteneur WAV. Sa reconnaissance : "
        f"{json.dumps(faits['format_reconnu'], ensure_ascii=False)}. "
        "Appelle `mesurer_media` puis `proposer_generateur`."
    )
    for _ in range(essais):
        COMPTEUR["appels"] += 1
        await service.get_chat_message_contents(chat_history=historique, settings=reglages,
                                                kernel=kernel)
        if etat["retenu"] is not None:
            break
        historique.add_user_message(
            "Aucune proposition acceptee. Les faits mesures sont inchanges : "
            f"{json.dumps(faits, ensure_ascii=False)[:900]}. Repropose.")
    if etat["retenu"] is None:
        raise RuntimeError(
            "le modele n'a propose aucun generateur reproduisant l'artefact -- passe rouge ; "
            f"propositions : {etat['propositions']}")
    return etat


_appels_avant_media = COMPTEUR["appels"]
_GENERATEUR = await demander_generateur(_ARTEFACT_A, _FAITS_A)
_APPELS_MEDIA = COMPTEUR["appels"] - _appels_avant_media

print(f"  modele servi        : {MODELE}")
print(f"  fonctions appelees  : {_GENERATEUR['appels']}")
print(f"  propositions        : {len(_GENERATEUR['propositions'])}")
for _i, _p in enumerate(_GENERATEUR["propositions"], 1):
    print(f"    {_i}. {_p['parametres']} -> identique={_p['verdict']['identique']} "
          f"sha={_p['verdict']['sha_propose']}")
print(f"  generateur retenu   : {_GENERATEUR['retenu']['parametres']}")
print(f"  sha artefact        : {_GENERATEUR['retenu']['verdict']['sha_artefact']}")

# Controle negatif 1 : le generateur retenu ne reproduit PAS le second media.
_VERDICT_B = identite_artefact(
    lambda: synthese_wav(**_GENERATEUR["retenu"]["parametres"]), _ARTEFACT_B)
print(f"  controle negatif B  : identique={_VERDICT_B['identique']} -- {_VERDICT_B['ecart']}")
assert not _VERDICT_B["identique"], "le generateur retenu reproduit aussi un autre media"

# Controle negatif 2 : un voisin proche du parametre retenu ne passe pas non plus.
_VOISIN = {**_GENERATEUR["retenu"]["parametres"],
           "frequence_hz": _GENERATEUR["retenu"]["parametres"]["frequence_hz"] + 1.0}
_VERDICT_VOISIN = identite_artefact(lambda: synthese_wav(**_VOISIN), _ARTEFACT_A)
print(f"  controle negatif +1 : identique={_VERDICT_VOISIN['identique']} "
      f"-- {_VERDICT_VOISIN['ecart']}")
assert not _VERDICT_VOISIN["identique"], "un parametre voisin passe le garde d'identite"

# La limite du garde, mesuree : il prouve la REPRODUCTION, pas la CAUSALITE.
_VERDICT_VIDE = identite_artefact(lambda: _ARTEFACT_A, _ARTEFACT_A)
assert _VERDICT_VIDE["identique"]
print("  limite du garde     : un generateur qui rend l'artefact tel quel passe aussi "
      f"(identique={_VERDICT_VIDE['identique']}) -- l'identite est necessaire, pas suffisante")

# La provenance entre dans les metadonnees du segment : c'est le champ que
# `MetadonneesSegment` declarait sans usage jusqu'ici.
_META_MEDIA = MetadonneesSegment(
    format=_FAITS_A["format_reconnu"]["format"],
    schema={"conteneur": _FAITS_A["conteneur"], "codec": _FAITS_A["codec"],
            "canaux": _FAITS_A["canaux"],
            "taux_echantillonnage_hz": _FAITS_A["taux_echantillonnage_hz"]},
    role="media",
    provenance={"generateur": "synthese_wav",
                "parametres": _GENERATEUR["retenu"]["parametres"],
                "verdict": _GENERATEUR["retenu"]["verdict"]},
    extras={"duree_s": _FAITS_A["duree_s"], "octets": len(_ARTEFACT_A)},
)
assert MetadonneesSegment.depuis_dict(
    json.loads(json.dumps(_META_MEDIA.vers_dict()))) == _META_MEDIA
print(f"  metadonnees         : {_META_MEDIA} ; cout {_META_MEDIA.cout()} o ; "
      f"aller-retour JSON exact")
print(f"  appels LLM          : {_APPELS_MEDIA} "
      f"(la synthese et les trois controles n'en font aucun)")

  modele servi        : qwen3.6-35b-a3b
  fonctions appelees  : ['mesurer_media']
  propositions        : 1
    1. {'frequence_hz': 500.0, 'amplitude_crete': 0.5, 'taux_echantillonnage': 8000, 'duree_s': 0.25} -> identique=True sha=3d3f56dcba5f76c3
  generateur retenu   : {'frequence_hz': 500.0, 'amplitude_crete': 0.5, 'taux_echantillonnage': 8000, 'duree_s': 0.25}
  sha artefact        : 3d3f56dcba5f76c3
  controle negatif B  : identique=False -- offset 46 : attendu 7461a00f000000007c0c7e188e23412d != obtenu 7461a00f000000007e18412d203b0040
  controle negatif +1 : identique=False -- offset 46 : attendu 7461a00f000000007e18412d203b0040 != obtenu 7461a00f000000008a18532d2f3bff3f
  limite du garde     : un generateur qui rend l'artefact tel quel passe aussi (identique=True) -- l'identite est necessaire, pas suffisante
  metadonnees         : <meta wav role=media schema=['canaux', 'codec', 'conteneur']> ; cout 511 o ; aller-retour JSON exact
  appels LLM          : 1 (la synthese et les t

### Le cas usuel : une sortie d'affichage, reproduite par un script

La section précédente a traité la famille où la question est **décidable** : un média dont les octets sont une fonction pure de quatre nombres. C'est un plancher, pas le cas usuel. Il est presque impossible de retrouver le générateur d'un WAV quelconque, et l'immense majorité des médias n'en a aucun de rejouable. Présenter ce cas comme *la* compression par provenance serait vendre un cas d'école.

Le cas usuel est celui d'une **sortie d'affichage** : une figure, un tableau, un document mis en forme. Un artefact volumineux dont la **logique d'assemblage** est petite. Le modèle n'a pas à deviner un générateur caché : il reçoit la géométrie **mesurée** des parties, et il **écrit le script** qui les réassemble. Le pipeline exécute ce script et compare les pixels.

Ce que la section fait, dans cet ordre :

1. le pipeline rend un artefact d'affichage à partir d'un script qu'il garde pour lui ;
2. il **mesure** les parties : composantes connexes par couleur, boîte englobante, aire en pixels, et si la boîte est entièrement remplie ;
3. le modèle **écrit un script** de reconstruction à partir de ces seules mesures ;
4. le pipeline l'**exécute** dans un espace de noms réduit, rend les pixels et les compare **un par un** ;
5. l'écart lui est rendu, situé au premier pixel différent, pour qu'il se corrige dans le même tour.

L'identité exigée ici est **au pixel**, pas à l'octet. Deux fichiers PNG différents peuvent rendre exactement la même image : un encodage n'est pas une fidélité, et le carnet nomme à chaque fois laquelle des deux il vérifie.

In [22]:
from PIL import Image, ImageDraw

LARGEUR_BARRE = 20


def rendre_artefact(barres: list[tuple[int, int, tuple[int, int, int]]],
                    taille: tuple[int, int] = (600, 240),
                    fond: tuple[int, int, int] = (255, 255, 255)) -> bytes:
    """Rend un PNG a partir de rectangles pleins : la geometrie determine les pixels.

    Pas de police, pas d'anti-aliasing, pas de transparence : les bords sont nets, donc
    une boite englobante mesuree suffit a rejouer la forme exactement. Une figure avec du
    texte, une courbe lissee ou un degrade sortirait de cette famille, et le carnet le dit
    plutot que de le laisser croire.
    """
    image = Image.new("RGB", taille, fond)
    pinceau = ImageDraw.Draw(image)
    for x0, hauteur, couleur in barres:
        pinceau.rectangle([x0, taille[1] - hauteur, x0 + LARGEUR_BARRE - 1, taille[1] - 1],
                          fill=couleur)
    tampon = io.BytesIO()
    image.save(tampon, format="PNG")
    return tampon.getvalue()


def _composantes(positions: set[tuple[int, int]]) -> list[tuple[list[int], int]]:
    """Les parties d'une meme couleur : composantes connexes 4-voisins, boite et aire.

    Une boite par COULEUR ne suffirait pas : deux barres de meme couleur se
    confondraient en une seule boite, et le modele n'aurait plus de quoi les replacer.
    """
    restants = set(positions)
    parties = []
    while restants:
        x, y = restants.pop()
        pile = [(x, y)]
        boite = [x, y, x, y]
        aire = 1
        while pile:
            cx, cy = pile.pop()
            for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                voisin = (cx + dx, cy + dy)
                if voisin in restants:
                    restants.discard(voisin)
                    pile.append(voisin)
                    aire += 1
                    boite[0] = min(boite[0], voisin[0])
                    boite[1] = min(boite[1], voisin[1])
                    boite[2] = max(boite[2], voisin[0])
                    boite[3] = max(boite[3], voisin[1])
        parties.append((boite, aire))
    return parties


def mesurer_formes(data: bytes) -> dict:
    """Les parties de l'artefact, MESUREES sur les pixels : rien n'est repris du script."""
    image = Image.open(io.BytesIO(data)).convert("RGB")
    largeur, hauteur = image.size
    brut = image.tobytes()
    compte = Counter(zip(brut[0::3], brut[1::3], brut[2::3]))
    fond = compte.most_common(1)[0][0]
    positions: dict[tuple[int, int, int], set[tuple[int, int]]] = {}
    for i in range(0, len(brut), 3):
        couleur = (brut[i], brut[i + 1], brut[i + 2])
        if couleur == fond:
            continue
        p = i // 3
        positions.setdefault(couleur, set()).add((p % largeur, p // largeur))
    formes = []
    for couleur, points in positions.items():
        for boite, aire in _composantes(points):
            formes.append({"couleur": list(couleur), "boite": boite, "pixels": aire,
                           "boite_pleine": aire == ((boite[2] - boite[0] + 1)
                                                    * (boite[3] - boite[1] + 1))})
    formes.sort(key=lambda f: (f["boite"][1], f["boite"][0]))
    return {"taille": [largeur, hauteur], "fond": list(fond), "formes": formes}


def identite_pixels(artefact: bytes, propose: bytes) -> dict:
    """L'identite au PIXEL, avec l'ecart situe au premier pixel qui differe."""
    attendu = Image.open(io.BytesIO(artefact)).convert("RGB")
    obtenu = Image.open(io.BytesIO(propose)).convert("RGB")
    if attendu.size != obtenu.size:
        return {"identique": False, "ecart": f"taille {obtenu.size} != {attendu.size}"}
    a, b = attendu.tobytes(), obtenu.tobytes()
    verdict = {"identique": a == b, "sha_pixels": hashlib.sha256(a).hexdigest()[:16],
               "pixels": attendu.size[0] * attendu.size[1], "ecart": "aucun"}
    if a == b:
        return verdict
    for i, (pa, pb) in enumerate(zip(a, b)):
        if pa != pb:
            p = i // 3
            verdict["ecart"] = (f"pixel ({p % attendu.size[0]}, {p // attendu.size[0]}) : "
                                f"attendu #{a[3 * p:3 * p + 3].hex()} != "
                                f"obtenu #{b[3 * p:3 * p + 3].hex()}")
            break
    return verdict


def executer_script(source: str, taille: tuple[int, int],
                    extra: dict | None = None) -> bytes:
    """Execute un script propose et rend ses octets. Espace de noms reduit.

    Ce n'est PAS un bac a sable de securite : le modele tourne en local et le carnet
    mesure une fidelite, pas une resistance. L'espace reduit sert a rendre le contrat
    explicite : le script recoit de quoi dessiner et la taille, il ne lit pas le disque.
    """
    espace = {"Image": Image, "ImageDraw": ImageDraw, "io": io, "taille": tuple(taille),
              "__builtins__": {"abs": abs, "enumerate": enumerate, "int": int, "len": len,
                               "list": list, "max": max, "min": min, "range": range,
                               "round": round, "sorted": sorted, "sum": sum, "tuple": tuple,
                               "zip": zip}}
    espace.update(extra or {})
    exec(compile(source, "<script propose>", "exec"), espace)
    rendre = espace.get("rendre")
    if not callable(rendre):
        raise ValueError("le script ne definit pas de fonction `rendre()`")
    produit = rendre()
    if not isinstance(produit, (bytes, bytearray)):
        raise ValueError(f"`rendre()` a rendu {type(produit).__name__}, pas des octets")
    return bytes(produit)


COULEURS_FIGURE = [(31, 119, 180), (255, 127, 14), (44, 160, 44), (214, 39, 40),
                   (148, 103, 189), (140, 86, 75)]
HAUTEURS_FIGURE = [40, 96, 61, 130, 82, 150, 55, 118, 74, 142, 90, 168,
                   47, 103, 68, 137, 86, 156, 59, 122, 79, 149, 93, 174]
BARRES_FIGURE = [(8 + i * 24, hauteur, COULEURS_FIGURE[i % len(COULEURS_FIGURE)])
                 for i, hauteur in enumerate(HAUTEURS_FIGURE)]

_ARTEFACT_FIGURE = rendre_artefact(BARRES_FIGURE)
_FIGURE = RACINE / "figure" / "tableau.png"
_FIGURE.parent.mkdir(exist_ok=True)
_FIGURE.write_bytes(_ARTEFACT_FIGURE)
_FAITS_FIGURE = mesurer_formes(_FIGURE.read_bytes())

# La mesure est une fonction des octets : deux lectures rendent les memes faits.
assert mesurer_formes(_ARTEFACT_FIGURE) == _FAITS_FIGURE
# Un PNG n'est pas l'identite : deux encodages du meme contenu rendent la meme image.
_reencode = io.BytesIO()
Image.open(io.BytesIO(_ARTEFACT_FIGURE)).save(_reencode, format="PNG", optimize=True)
_REENCODE = _reencode.getvalue()
assert _REENCODE != _ARTEFACT_FIGURE, "les deux encodages coincident, le controle est vide"
assert identite_pixels(_ARTEFACT_FIGURE, _REENCODE)["identique"]
# ... et une image differente ne passe pas, meme a taille egale.
assert not identite_pixels(_ARTEFACT_FIGURE, rendre_artefact(BARRES_FIGURE[:-1]))["identique"]

print(f"artefact : {len(_ARTEFACT_FIGURE)} o de PNG, {_FAITS_FIGURE['taille'][0]}x"
      f"{_FAITS_FIGURE['taille'][1]} pixels, fond {tuple(_FAITS_FIGURE['fond'])}")
print(f"  reencodage : {len(_REENCODE)} o, octets differents mais "
      f"{_FAITS_FIGURE['taille'][0] * _FAITS_FIGURE['taille'][1]} pixels identiques")
print(f"  parties mesurees : {len(_FAITS_FIGURE['formes'])}, toutes pleines = "
      f"{all(f['boite_pleine'] for f in _FAITS_FIGURE['formes'])}")
for _forme in _FAITS_FIGURE["formes"][:3]:
    print(f"    {tuple(_forme['couleur'])} boite {_forme['boite']} "
          f"aire {_forme['pixels']} pixels")
print(f"  faits remis au modele : {len(json.dumps(_FAITS_FIGURE, ensure_ascii=False))} caracteres")

# Ce que le meme contenu coute sur un grand canevas : la ou la recette gagne.
_GRAND = rendre_artefact(BARRES_FIGURE, taille=(3000, 2000))
print(f"  meme contenu en 3000x2000 : {len(_GRAND)} o de PNG "
      f"({len(_GRAND) / len(_ARTEFACT_FIGURE):.1f} fois le petit)")

artefact : 1197 o de PNG, 600x240 pixels, fond (255, 255, 255)
  reencodage : 987 o, octets differents mais 144000 pixels identiques
  parties mesurees : 24, toutes pleines = True
    (140, 86, 75) boite [560, 66, 579, 239] aire 3480 pixels
    (140, 86, 75) boite [272, 72, 291, 239] aire 3360 pixels
    (140, 86, 75) boite [416, 84, 435, 239] aire 3120 pixels
  faits remis au modele : 2386 caracteres
  meme contenu en 3000x2000 : 22723 o de PNG (19.0 fois le petit)


### Le modèle écrit le script, le pipeline l'exécute et compare les pixels

Le plugin ne rend pas l'artefact : il rend le **verdict**, écart situé au premier pixel différent compris, pour que le modèle puisse se corriger dans le même tour. Un script n'est accepté que si les pixels rejoués sont identiques à l'artefact. Il n'y a pas de tolérance, pas de seuil de similarité, pas de « presque ».

Trois relances sont permises, pas davantage : passé ce point la cellule échoue. Un repli silencieux fabriquerait un résultat que rien n'a produit, et c'est exactement la faute que le carnet refuse ailleurs.

Les contrôles qui suivent disent ce que le garde vaut : un décalage d'un seul pixel le fait échouer, et un script qui rend l'artefact tel quel le fait passer. L'identité prouve donc la **reproduction**, jamais la **causalité** : c'est la limite du garde, mesurée et non supposée.

In [23]:
class PluginScriptArtefact:
    """Le modele ECRIT le script ; le pipeline l'execute et compare les pixels."""

    def __init__(self, etat: dict):
        self.etat = etat

    @kernel_function(
        name="mesurer_formes",
        description=("Rend les parties mesurees de l'artefact : taille de l'image, couleur "
                     "de fond, puis pour chaque partie sa couleur RVB, sa boite englobante "
                     "[x0, y0, x1, y1] et son aire en pixels."),
    )
    def mesurer_formes(self) -> str:
        self.etat["appels"].append("mesurer_formes")
        return json.dumps(self.etat["faits"], ensure_ascii=False)

    @kernel_function(
        name="proposer_script",
        description=("Propose le source Python d'un script qui definit une fonction "
                     "`rendre()` rendant les octets PNG de l'artefact. Le script recoit "
                     "`Image`, `ImageDraw`, `io` et `taille` ; il n'a pas le droit "
                     "d'importer. Le pipeline l'execute et compare les pixels un par un."),
    )
    def proposer_script(self, source: str) -> str:
        self.etat["tentatives"].append(source)
        try:
            produit = executer_script(source, tuple(self.etat["faits"]["taille"]))
        except Exception as erreur:  # un script casse est un verdict, pas un plantage
            verdict = {"identique": False,
                       "ecart": f"script en echec : {type(erreur).__name__}: {erreur}"}
        else:
            verdict = identite_pixels(self.etat["artefact"], produit)
        self.etat["verdicts"].append(verdict)
        if verdict["identique"]:
            self.etat["retenu"] = {"source": source, "verdict": verdict}
            return (f"accepte : les {verdict['pixels']} pixels rejoues sont identiques "
                    f"(sha {verdict['sha_pixels']})")
        return (f"refuse : {verdict['ecart']}. Reprends les mesures et repropose. "
                "`ImageDraw.rectangle` inclut ses deux extremites, et `y` croit vers le bas.")


async def demander_script(artefact: bytes, faits: dict, essais: int = 3) -> dict:
    """La boucle du cas usuel : le modele ecrit, le pipeline execute et compare."""
    etat = {"artefact": artefact, "faits": faits, "tentatives": [], "verdicts": [],
            "retenu": None, "appels": []}
    kernel = Kernel()
    kernel.add_service(OpenAIChatCompletion(ai_model_id=MODELE, async_client=aoai,
                                            service_id=SERVICE))
    kernel.add_plugin(PluginScriptArtefact(etat), plugin_name="figure")
    reglages = OpenAIPromptExecutionSettings(
        max_tokens=3000, temperature=0.0,
        function_choice_behavior=FunctionChoiceBehavior.Auto(
            auto_invoke=True, maximum_auto_invoke_attempts=6),
    )
    service = kernel.get_service(SERVICE)
    historique = ChatHistory()
    historique.add_system_message(
        "Tu ecris le SCRIPT Python qui reconstruit une image a partir des mesures de ses "
        "parties. Tu appelles `mesurer_formes`, puis `proposer_script` avec le source "
        "complet du script. Le script definit `rendre()` qui rend les octets PNG et "
        "n'importe rien : il dispose de `Image`, `ImageDraw`, `io` et `taille`. "
        "`ImageDraw.rectangle` prend [x0, y0, x1, y1] et inclut les deux extremites ; "
        "`y` croit vers le bas. Si un script est refuse, lis le pixel cite et corrige."
    )
    historique.add_user_message(
        "L'image mesuree est un tableau de barres pleines, sans texte ni degrade. Chaque "
        "partie mesuree est exactement un rectangle plein : sa boite suffit a la replacer, "
        "et la couleur du fond complete l'image. Appelle `mesurer_formes` puis "
        "`proposer_script`."
    )
    for _ in range(essais):
        COMPTEUR["appels"] += 1
        await service.get_chat_message_contents(chat_history=historique, settings=reglages,
                                                kernel=kernel)
        if etat["retenu"] is not None:
            break
        historique.add_user_message(
            "Aucun script accepte. Les mesures sont inchangees : "
            f"{json.dumps(faits, ensure_ascii=False)}. Repropose en corrigeant le pixel cite.")
    if etat["retenu"] is None:
        raise RuntimeError(
            f"le modele n'a ecrit aucun script reproduisant l'artefact -- passe rouge ; "
            f"verdicts : {etat['verdicts']}")
    return etat


_appels_avant_script = COMPTEUR["appels"]
_SCRIPT = await demander_script(_ARTEFACT_FIGURE, _FAITS_FIGURE)
_APPELS_SCRIPT = COMPTEUR["appels"] - _appels_avant_script
_SOURCE_RETENUE = _SCRIPT["retenu"]["source"]
_OCTETS_SCRIPT = len(_SOURCE_RETENUE.encode("utf-8"))

print(f"  modele servi        : {MODELE}")
print(f"  fonctions appelees  : {_SCRIPT['appels']}")
print(f"  tentatives          : {len(_SCRIPT['tentatives'])}")
for _i, _verdict in enumerate(_SCRIPT["verdicts"], 1):
    print(f"    {_i}. identique={_verdict['identique']} -- {_verdict['ecart'][:110]}")
print(f"  script retenu       : {_OCTETS_SCRIPT} o, "
      f"{_SOURCE_RETENUE.count(chr(10)) + 1} lignes")
print(f"  sha des pixels      : {_SCRIPT['retenu']['verdict']['sha_pixels']}")
print()
print(_SOURCE_RETENUE)

# Rejouabilite : deux executions du script retenu rendent les memes octets.
_PRODUIT_1 = executer_script(_SOURCE_RETENUE, tuple(_FAITS_FIGURE["taille"]))
_PRODUIT_2 = executer_script(_SOURCE_RETENUE, tuple(_FAITS_FIGURE["taille"]))
assert _PRODUIT_1 == _PRODUIT_2, "le script retenu n'est pas rejouable a l'identique"

# Controle negatif : un decalage d'un seul pixel fait echouer le garde.
_image = Image.open(io.BytesIO(_PRODUIT_1)).convert("RGB")
_decalee = Image.new("RGB", _image.size, (0, 0, 0))
_decalee.paste(_image, (1, 0))
_tampon_decale = io.BytesIO()
_decalee.save(_tampon_decale, format="PNG")
_VERDICT_DECALE = identite_pixels(_ARTEFACT_FIGURE, _tampon_decale.getvalue())
print(f"  controle negatif (decalage 1 px) : identique={_VERDICT_DECALE['identique']} "
      f"-- {_VERDICT_DECALE['ecart']}")
assert not _VERDICT_DECALE["identique"], "un decalage d'un pixel passe le garde"

# Limite du garde, mesuree : un script qui rend l'artefact tel quel passe aussi.
_VERDICT_COPIE = identite_pixels(
    _ARTEFACT_FIGURE,
    executer_script("def rendre():\n    return artefact_octets",
                    tuple(_FAITS_FIGURE["taille"]), {"artefact_octets": _ARTEFACT_FIGURE}))
assert _VERDICT_COPIE["identique"]
print("  limite du garde     : un script qui rend l'artefact tel quel passe aussi "
      f"(identique={_VERDICT_COPIE['identique']}) -- l'identite est necessaire, pas suffisante")

print()
print(f"  artefact            : {len(_ARTEFACT_FIGURE)} o de PNG")
print(f"  script retenu       : {_OCTETS_SCRIPT} o")
print(f"  rapport artefact/script : {len(_ARTEFACT_FIGURE) / _OCTETS_SCRIPT:.2f}")
print(f"  meme contenu en 3000x2000 : {len(_GRAND)} o de PNG pour le meme script "
      f"(rapport {len(_GRAND) / _OCTETS_SCRIPT:.1f})")
print(f"  appels LLM          : {_APPELS_SCRIPT} "
      f"(l'execution, la comparaison et les controles n'en font aucun)")

  modele servi        : qwen3.6-35b-a3b
  fonctions appelees  : ['mesurer_formes']
  tentatives          : 2
    1. identique=False -- script en echec : TypeError: color must be int or tuple
    2. identique=True -- aucun
  script retenu       : 1399 o, 34 lignes
  sha des pixels      : 4ae7711c22993e57

def rendre():
    img = Image.new("RGB", taille, (255, 255, 255))
    d = ImageDraw.Draw(img)
    barres = [
        ((140, 86, 75), [560, 66, 579, 239]),
        ((140, 86, 75), [272, 72, 291, 239]),
        ((140, 86, 75), [416, 84, 435, 239]),
        ((140, 86, 75), [128, 90, 147, 239]),
        ((214, 39, 40), [512, 91, 531, 239]),
        ((214, 39, 40), [224, 98, 243, 239]),
        ((214, 39, 40), [368, 103, 387, 239]),
        ((214, 39, 40), [80, 110, 99, 239]),
        ((255, 127, 14), [464, 118, 483, 239]),
        ((255, 127, 14), [176, 122, 195, 239]),
        ((255, 127, 14), [320, 137, 339, 239]),
        ((255, 127, 14), [32, 144, 51, 239]),
        ((148, 103, 189), [

### Lecture du résultat

Le cas usuel se comporte autrement que le cas décidable, et c'est instructif.

**Le modèle écrit du code, et ce code est vérifié.** Ce n'est plus un jeu de paramètres : c'est un script complet, exécuté, dont les pixels sont comparés un par un. Le garde attrape un décalage d'un seul pixel et nomme l'endroit où il tombe : la vérification est située, pas binaire.

**Sur ce petit artefact, le script n'est pas plus petit que le PNG.** Un rendu en aplats est précisément ce que le format PNG compresse bien : annoncer un gain ici serait malhonnête, et le carnet publie les deux nombres au lieu de choisir celui qui l'arrange. Ce que la section établit n'est donc pas un taux de compression, c'est une **substituabilité vérifiée** : l'artefact est remplaçable par un script dont on sait prouver qu'il le reproduit.

**Le rapport bascule quand le canevas grandit.** Le même contenu rendu en 3000x2000 coûte bien plus de PNG pour le même script, parce que le script décrit la géométrie et non la surface. C'est là que la recette gagne : quand l'artefact est cher à encoder relativement à sa description, ce qui est le cas des sorties d'affichage volumineuses, pas des aplats de 600x240.

**La limite est la même qu'ailleurs, et elle est mesurée ici.** Un script qui rend l'artefact tel quel passe le garde : l'identité prouve la reproduction, jamais la causalité. Et la famille est déclarée : rectangles pleins, sans texte, sans courbe lissée, sans dégradé. Une figure qui contient du texte ou de l'anti-aliasing demanderait une comparaison tolérante, et cette tolérance n'est pas démontrée ici.

## Ce que ce carnet a établi

| Acquis | Où il se vérifie |
|---|---|
| Le modèle choisit et **exécute** réellement des fonctions, sous Semantic Kernel | le compteur d'appels du triage, et la trace des décisions |
| Les opérations sont **inversibles**, et chaque inverse est exercé | la vérification de contrat sur les quatre codecs, puis l'égalité binaire |
| La structure des formats est **présentée** au modèle, pas devinée | `structure_interne` : colonnes, clés, niveaux, titres |
| La couverture est un **invariant**, pas une espérance | le contrôle négatif qui la fait échouer, puis la vérification en fin de passe |
| La décompression **n'appelle personne** | le compteur remis à zéro, puis relu à zéro |
| La comparaison est **publiable dans les deux sens** | les trois totaux, recette et dictionnaire compris |
| Un segment peut porter des **repères** et des **métadonnées** | `Repere` (quatre types, contrôle négatif à l'appui) et `MetadonneesSegment` (aller-retour JSON, coût affiché) |
| Le format est reconnu sur les **octets**, jamais sur le nom | le même CSV renommé en `.dat` reste un CSV, et son menu de découpage ne change pas |
| Un sérialiseur n'est un générateur que s'il **reproduit** l'artefact | `identite_artefact`, avec l'écart situé ; identité binaire ou seulement sémantique, et le carnet dit laquelle |
| Un média déterministe livre sa **provenance** vérifiable | le générateur proposé par le modèle reproduit l'artefact au bit près ; deux contrôles négatifs montrent que le garde discrimine, et un troisième mesure sa limite |
| Un artefact d'affichage se remplace par le **script** qui le reproduit | le modèle écrit le script à partir des parties mesurées, le pipeline l'exécute et compare les pixels un par un ; un décalage d'un pixel échoue |
| L'entonnoir est **mesuré dans les deux sens** | les trois routes par fichier, l'arbitrage, et le contrôle positif qui prouve que le mécanisme sait gagner |

### Ce que ce carnet ne prétend pas

Le corpus fait huit fichiers. Un gain ou une perte sur huit fichiers ne s'extrapole pas à un corpus réel : les codecs de la bibliothèque standard ont des fenêtres de plusieurs mégaoctets, et un corpus plus gros leur donne un avantage que ce banc ne leur donne pas encore. Le résultat publié vaut pour **ce** corpus, mesuré ici, et la cellule de comparaison donne de quoi le réfuter en changeant le corpus.

Le triage lui-même n'est pas évalué à sa juste valeur par ce banc : le carnet mesure ce que le modèle a choisi, il ne mesure pas ce qu'un triage parfait aurait rapporté. L'entonnoir de la dernière section comble une partie de cet écart — il exécute le découpage systématique que le triage aurait pu choisir, et il le chiffre. Sur ce corpus-ci le découpage perd, et sur un CSV large à colonnes faiblement cardinales il gagne largement : les deux mesures sont publiées ensemble, dans la même cellule, et c'est la forme des données qui les sépare.

Une chose était écrite ici comme **non** démontrée : inférer le code qui a produit un média et le proposer après vérification d'identité. Elle l'est désormais, sur la famille où la question est décidable — un média dont les octets sont une fonction pure de quatre nombres. Le modèle a lu les faits mesurés sur les octets et proposé les paramètres du générateur ; le pipeline a synthétisé et comparé au bit près, et deux contrôles négatifs montrent que le garde discrimine.

Ce qui reste hors de portée est mesuré dans la même section : le garde prouve qu'un code **reproduit** l'artefact, pas qu'il l'a produit, et un média non déterministe — une image de diffusion, un son de modèle génératif — n'a pas de générateur rejouable à qui appliquer ce garde.

### Pour continuer

- changer la **forme** du corpus — fichiers plus larges, colonnes faiblement cardinales — et regarder l'arbitrage basculer vers le découpage ;
- comparer deux modèles servis par le même endpoint sur le même corpus : mêmes décisions ou non ;
- instrumenter le coût en jetons du triage, qui n'est pas compté ici et qui déciderait, en pratique, de l'intérêt du procédé.

Le carnet [07](Compression-07-Triage-LLM-Operations-Reelles-Python.ipynb) reste la première expérience, avec ses schémas d'outils écrits à la main ; celui-ci en est l'approfondissement sous harnais, avec la couverture prouvée.

## À retenir

- Un codec n'est une **primitive fiable** que si son inverse a été éprouvé sur
  vide, texte et binaire **avant** d'être exposé au modèle : la symétrie
  `CODECS` / inverse est la même décision écrite deux fois.
- L'**invariant de couverture** doit échouer quand il le faut. Une recette qui
  n'a jamais levé `CouvertureIncomplete` n'est pas un invariant, c'est une
  fonction qui rend `True` : le contrôle négatif (colonnes oubliées) compte
  autant que le positif.
- La thèse se vérifie **à la décompression** : zéro appel au modèle, et égalité
  binaire avec les originaux. La recette rejouée à l'envers est la preuve que
  le triage n'a rien inventé.
- L'**entonnoir** renverse la répartition du travail : le modèle fournit les
  informations de base (format, schéma, repères), l'automatisme fait le gros
  œuvre. C'est la même discipline que le carnet 09.


## Vérifiez votre compréhension

- Pourquoi la reconnaissance du format se fait-elle **sur les octets**
  (signature) et non sur l'extension ? Qu'arrive-t-il à un CSV renommé `.bin`
  si l'on ne regardait que le suffixe ?
- Le compteur d'appels est incrémenté **juste avant** l'envoi réel. Pourquoi
  ce détail de placement est-il ce qui autorise à écrire « N appels **réels** »
  plutôt qu'un nombre simplement promis ?
- Le codec à dictionnaire est appris **une fois** sur le corpus puis transmis.
  En quoi cela change-t-il le contrat de chaque segment par rapport aux trois
  codecs « de zéro » ?


## Pour aller plus loin

- Le **carnet 07** (triage main : schémas d'outils écrits à la main en JSON,
  requêtes postées en `urllib`) : ce que le harnais Semantic Kernel **achète**,
  et ce qu'il ne change **pas** à la thèse.
- Le **carnet 09** (assemblage par proposition) : l'entonnoir poussé à son
  terme — le modèle ne fournit que le digest, le pipeline scripte et juge.
- L'**Exercice 3** (borne « meilleur codec par fichier ») : la borne la plus
  honnête à opposer au triage, et ce qu'elle dirait sur un corpus homogène.


In [24]:
# Le repertoire de travail est propre a cette execution : on le retire.
shutil.rmtree(RACINE, ignore_errors=True)
print("repertoire de travail retire")
_total_appels = COMPTEUR['appels_tri'] + COMPTEUR['appels']
_appels_infos_base = COMPTEUR['appels'] - _APPELS_MEDIA - _APPELS_SCRIPT
print(f"recapitulatif : {_total_appels} appels LLM au total "
      f"(dont {COMPTEUR['appels_tri']} de triage, {_appels_infos_base} pour les infos de base "
      f"et {_APPELS_MEDIA + _APPELS_SCRIPT} pour la provenance des artefacts "
      f"(media et figure)), 0 pendant la decompression")


repertoire de travail retire
recapitulatif : 22 appels LLM au total (dont 19 de triage, 1 pour les infos de base et 2 pour la provenance des artefacts (media et figure)), 0 pendant la decompression
